# Kaapi Copilot — Agentic Storefront for Kaapi Roasters

A working prototype of a conversational, agent-to-agent-transactable D2C storefront:

- **Journey A** — human buyer chats with a Claude-powered shopping agent that recommends products, proposes one catalog-defined upsell, builds a **Purchase Mandate**, and only calls Razorpay after an explicit buyer confirmation.
- **Journey B** — the identical catalog/cart/checkout exposed as MCP-style tools so an external AI agent can shop and pay with no human in the loop.
- A **Guardrail / Mandate Engine** gates every payment (spend caps, catalog price re-verification, explicit confirmation) and writes a **hash-chained audit trail** before any money call.
- Razorpay (test mode) handles orders/payment links; a verified webhook drives the order state machine, including a deliberate **UPI decline failure** handled gracefully.

## Build order (Section 10)
1. Catalog service + seed data
2. Guardrail / Mandate Engine + hash-chained audit log
3. Razorpay integration (test mode): order → payment link → webhook → paid
4. Graceful failure flow (UPI decline + guardrail rejection)
5. Journey A conversational agent (Claude, tool use)
6. Journey B — MCP tool exposure for an external AI buyer agent
7. Dashboard + README

## 1. Project setup — check environment
Verify required secrets are available before wiring live integrations, and scaffold the project directory structure.

## 1. Project Structure

Scaffold the full repository layout for Kaapi Copilot: backend (FastAPI-based Python app with provider/agent abstractions), frontend (static demo UI), config, and docs folders. This runs as a standalone application on disk — independent of the notebook kernel — so the notebook is used only for documentation/build-log purposes going forward.

In [23]:
mkdir -p kaapi_copilot/backend/app/{core,models,providers/payment,providers/agent,services,api,data}
mkdir -p kaapi_copilot/frontend
mkdir -p kaapi_copilot/docs
mkdir -p kaapi_copilot/tests

find kaapi_copilot -type d | sort


SyntaxError: invalid syntax (1248200643.py, line 1)

## 2. Configuration & Environment Handling

Central `config.py` reads `PAYMENT_MODE` (`mock` | `razorpay`), `AGENT_MODE` (`mock` | `anthropic`), and all secrets (`ANTHROPIC_API_KEY`, `RAZORPAY_KEY_ID`, `RAZORPAY_KEY_SECRET`, `RAZORPAY_WEBHOOK_SECRET`), plus guardrail limits (session/transaction spend caps). Missing secrets automatically force the corresponding mode to `mock` — this is the single source of truth the provider factories consult.

In [1]:
config_py = '''"""
Central configuration for Kaapi Copilot.
Reads env vars, decides mock vs real mode for payments and AI agent.
Missing secrets automatically force the corresponding subsystem into mock mode.
"""
import os
from dataclasses import dataclass, field


def _env(name: str, default: str = "") -> str:
    return os.environ.get(name, default).strip()


@dataclass
class Settings:
    # ---- Payment provider ----
    payment_mode_requested: str = field(default_factory=lambda: _env("PAYMENT_MODE", "mock").lower())
    razorpay_key_id: str = field(default_factory=lambda: _env("RAZORPAY_KEY_ID"))
    razorpay_key_secret: str = field(default_factory=lambda: _env("RAZORPAY_KEY_SECRET"))
    razorpay_webhook_secret: str = field(default_factory=lambda: _env("RAZORPAY_WEBHOOK_SECRET"))

    # ---- AI agent provider ----
    agent_mode_requested: str = field(default_factory=lambda: _env("AGENT_MODE", "mock").lower())
    groq_api_key: str = field(default_factory=lambda: _env("GROQ_API_KEY"))
    groq_model: str = field(default_factory=lambda: _env("GROQ_MODEL", "llama-3.3-70b-versatile"))

    # ---- Guardrail limits (paise) ----
    session_spend_cap_paise: int = int(_env("SESSION_SPEND_CAP_PAISE", "500000"))       # ₹5,000
    transaction_spend_cap_paise: int = int(_env("TRANSACTION_SPEND_CAP_PAISE", "300000"))  # ₹3,000
    max_upsells_per_turn: int = 1

    # ---- Misc ----
    cart_hold_minutes: int = 10
    db_path: str = field(default_factory=lambda: _env("KAAPI_DB_PATH", "kaapi_copilot.db"))
    webhook_base_url: str = field(default_factory=lambda: _env("WEBHOOK_BASE_URL", "http://localhost:8000"))

    @property
    def payment_mode(self) -> str:
        """Fall back to mock if razorpay requested but secrets missing."""
        if self.payment_mode_requested == "razorpay" and self.razorpay_key_id and self.razorpay_key_secret:
            return "razorpay"
        return "mock"

    @property
    def agent_mode(self) -> str:
        """Fall back to mock if groq requested but key missing."""
        if self.agent_mode_requested == "groq" and self.groq_api_key:
            return "groq"
        return "mock"

    def summary(self) -> dict:
        return {
            "payment_mode_requested": self.payment_mode_requested,
            "payment_mode_effective": self.payment_mode,
            "razorpay_keys_present": bool(self.razorpay_key_id and self.razorpay_key_secret),
            "agent_mode_requested": self.agent_mode_requested,
            "agent_mode_effective": self.agent_mode,
            "groq_key_present": bool(self.groq_api_key),
            "session_spend_cap_paise": self.session_spend_cap_paise,
            "transaction_spend_cap_paise": self.transaction_spend_cap_paise,
        }


settings = Settings()
'''

with open("kaapi_copilot/backend/app/core/config.py", "w") as f:
    f.write(config_py)

with open("kaapi_copilot/backend/app/core/__init__.py", "w") as f:
    f.write("")
with open("kaapi_copilot/backend/app/__init__.py", "w") as f:
    f.write("")

import subprocess, sys
sys.path.insert(0, "kaapi_copilot/backend")
for mod in list(sys.modules):
    if mod.startswith("app."):
        del sys.modules[mod]
from app.core.config import settings
print(settings.summary())


{'payment_mode_requested': 'mock', 'payment_mode_effective': 'mock', 'razorpay_keys_present': False, 'agent_mode_requested': 'mock', 'agent_mode_effective': 'mock', 'groq_key_present': False, 'session_spend_cap_paise': 500000, 'transaction_spend_cap_paise': 300000}


## 3. Data Models

Define dataclasses for the core domain objects: `Product`, `CartItem`, `Cart`, `PolicyCheck`, `PurchaseMandate`, `Order`, `AuditEvent`, and `PaymentResult`. These are provider-agnostic — used identically by mock and real payment/agent implementations.

In [3]:
models_py = '''"""
Core domain models for Kaapi Copilot. Provider-agnostic dataclasses shared by
mock and real payment/agent implementations, and by both Journey A and B.
"""
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Optional
import uuid


def _now() -> str:
    return datetime.now(timezone.utc).isoformat()


def _id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:12]}"


@dataclass
class Product:
    sku: str
    name: str
    price_paise: int
    category: str
    description: str = ""
    upsell_pairs: list = field(default_factory=list)  # list of SKUs

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class CartItem:
    sku: str
    name: str
    qty: int
    unit_price_paise: int

    @property
    def subtotal_paise(self) -> int:
        return self.qty * self.unit_price_paise

    def to_dict(self) -> dict:
        d = asdict(self)
        d["subtotal_paise"] = self.subtotal_paise
        return d


@dataclass
class Cart:
    session_id: str
    buyer_ref: str
    items: list = field(default_factory=list)  # list[CartItem]
    created_at: str = field(default_factory=_now)
    upsell_offered_skus: list = field(default_factory=list)

    @property
    def total_paise(self) -> int:
        return sum(i.subtotal_paise for i in self.items)

    def to_dict(self) -> dict:
        return {
            "session_id": self.session_id,
            "buyer_ref": self.buyer_ref,
            "items": [i.to_dict() for i in self.items],
            "total_paise": self.total_paise,
            "created_at": self.created_at,
            "upsell_offered_skus": self.upsell_offered_skus,
        }


@dataclass
class PolicyCheck:
    rule: str
    status: str  # "pass" | "fail"
    limit_paise: Optional[int] = None
    detail: str = ""

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class Confirmation:
    method: str = "pending"  # "buyer_tap" | "mcp_confirm_and_pay" | "pending"
    status: str = "pending"  # "pending" | "confirmed" | "rejected"
    confirmed_at: Optional[str] = None

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class PurchaseMandate:
    mandate_id: str
    session_id: str
    buyer_ref: str
    items: list  # list[CartItem]
    currency: str
    total_paise: int
    rationale: str
    policy_checks: list  # list[PolicyCheck]
    status: str  # "pending" | "confirmed" | "blocked" | "paid" | "payment_failed"
    created_at: str = field(default_factory=_now)
    confirmation: Confirmation = field(default_factory=Confirmation)
    order_id: Optional[str] = None
    block_reason: Optional[str] = None

    def to_dict(self) -> dict:
        return {
            "mandate_id": self.mandate_id,
            "session_id": self.session_id,
            "created_at": self.created_at,
            "buyer_ref": self.buyer_ref,
            "items": [i.to_dict() for i in self.items],
            "currency": self.currency,
            "total_paise": self.total_paise,
            "rationale": self.rationale,
            "policy_checks": [c.to_dict() for c in self.policy_checks],
            "confirmation": self.confirmation.to_dict(),
            "status": self.status,
            "order_id": self.order_id,
            "block_reason": self.block_reason,
        }


@dataclass
class Order:
    order_id: str
    mandate_id: str
    session_id: str
    total_paise: int
    currency: str
    status: str  # "created" | "paid" | "payment_failed"
    payment_link_id: Optional[str] = None
    payment_link_url: Optional[str] = None
    created_at: str = field(default_factory=_now)
    updated_at: str = field(default_factory=_now)
    failure_reason: Optional[str] = None

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class PaymentResult:
    order_id: str
    payment_link_id: str
    payment_link_url: str
    status: str  # "created"
    raw: dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class AuditEvent:
    event_id: str
    ts: str
    event_type: str
    session_id: str
    payload: dict
    prev_hash: str
    hash: str

    def to_dict(self) -> dict:
        return asdict(self)


def new_mandate_id() -> str:
    return _id("mnd")


def new_session_id() -> str:
    return _id("sess")


def new_order_id() -> str:
    return _id("order")


def new_event_id() -> str:
    return _id("evt")
'''

with open("kaapi_copilot/backend/app/models/__init__.py", "w") as f:
    f.write("")
with open("kaapi_copilot/backend/app/models/domain.py", "w") as f:
    f.write(models_py)

from app.models.domain import Product, Cart, CartItem, PurchaseMandate
p = Product(sku="kr-filter-500", name="Filter Coffee Powder 500g", price_paise=45000, category="powder", upsell_pairs=["kr-steel-filter"])
print(p.to_dict())


{'sku': 'kr-filter-500', 'name': 'Filter Coffee Powder 500g', 'price_paise': 45000, 'category': 'powder', 'description': '', 'upsell_pairs': ['kr-steel-filter']}


## 4. Mock Merchant & Catalog Data

Deterministic seed catalog for Kaapi Roasters (7 SKUs with real upsell/cross-sell pairs) plus a `CatalogService` exposing `list_products`, `get_product`, and catalog-price re-verification — the single source of truth both journeys and the guardrail engine read from.

In [4]:
catalog_data_py = '''"""
Seed catalog for Kaapi Roasters. Deterministic — same data every run.
Every product carries category + upsell_pairs so recommendations are
data-driven, never improvised by the LLM.
"""
from app.models.domain import Product

MERCHANT = {
    "merchant_id": "kaapi-roasters",
    "name": "Kaapi Roasters",
    "description": "Small D2C filter-coffee brand.",
    "onboarded_at": "2026-08-26T00:00:00Z",
}

SEED_PRODUCTS = [
    Product("kr-filter-500", "Filter Coffee Powder, 500g", 45000, "powder",
            "Classic South Indian filter coffee blend.", ["kr-steel-filter"]),
    Product("kr-arabica-250", "Single-Origin Arabica Beans, 250g", 38000, "beans",
            "Light-roast single-origin arabica.", ["kr-dripper"]),
    Product("kr-steel-filter", "South Indian Steel Filter Set", 65000, "brew-gear",
            "Traditional stainless steel filter set.", []),
    Product("kr-dripper", "Pour-over Dripper", 90000, "brew-gear",
            "Ceramic pour-over dripper.", []),
    Product("kr-subscription", "Monthly Coffee Subscription (2 bags)", 70000, "subscription",
            "Two bags delivered monthly.", []),
    Product("kr-frother", "Milk Frother", 55000, "accessory",
            "Handheld milk frother.", []),
    Product("kr-filters-100", "Reusable Filter Papers, pack of 100", 25000, "consumable",
            "Pack of 100 reusable filter papers.", ["kr-dripper"]),
]

CATALOG_BY_SKU = {p.sku: p for p in SEED_PRODUCTS}
'''

with open("kaapi_copilot/backend/app/data/__init__.py", "w") as f:
    f.write("")
with open("kaapi_copilot/backend/app/data/catalog_data.py", "w") as f:
    f.write(catalog_data_py)

catalog_service_py = '''"""
Catalog Service — single source of truth for product data. Exposed to both
the conversational agent (Journey A) and the MCP tool surface (Journey B).
"""
from typing import Optional
from app.data.catalog_data import CATALOG_BY_SKU, SEED_PRODUCTS, MERCHANT
from app.models.domain import Product


class CatalogService:
    def list_products(self, category: Optional[str] = None) -> list:
        products = SEED_PRODUCTS
        if category:
            products = [p for p in products if p.category == category]
        return [p.to_dict() for p in products]

    def get_product(self, sku: str) -> Optional[dict]:
        p = CATALOG_BY_SKU.get(sku)
        return p.to_dict() if p else None

    def get_price_paise(self, sku: str) -> Optional[int]:
        """Authoritative price lookup — never trust an LLM-stated price."""
        p = CATALOG_BY_SKU.get(sku)
        return p.price_paise if p else None

    def get_upsell_for(self, sku: str) -> Optional[dict]:
        p = CATALOG_BY_SKU.get(sku)
        if not p or not p.upsell_pairs:
            return None
        return self.get_product(p.upsell_pairs[0])

    def search(self, query: str) -> list:
        q = query.lower()
        return [p.to_dict() for p in SEED_PRODUCTS
                if q in p.name.lower() or q in p.category.lower() or q in p.description.lower()]

    def merchant_info(self) -> dict:
        return MERCHANT


catalog_service = CatalogService()
'''

with open("kaapi_copilot/backend/app/services/__init__.py", "w") as f:
    f.write("")
with open("kaapi_copilot/backend/app/services/catalog_service.py", "w") as f:
    f.write(catalog_service_py)

from app.services.catalog_service import catalog_service
print(len(catalog_service.list_products()), "products loaded")
print(catalog_service.get_upsell_for("kr-filter-500"))


7 products loaded

{'sku': 'kr-steel-filter', 'name': 'South Indian Steel Filter Set', 'price_paise': 65000, 'category': 'brew-gear', 'description': 'Traditional stainless steel filter set.', 'upsell_pairs': []}


## 5–6. Payment Provider Abstraction + Mock Razorpay Implementation

Define a `PaymentProvider` interface (`create_order`, `create_payment_link`, `fetch_payment_link`, `capture_payment`, `fetch_payment`, `create_refund`) so business logic never depends on Razorpay directly. `MockPaymentProvider` simulates order/payment-link creation deterministically and lets the demo trigger `success@razorpay` or `failure@razorpay` outcomes on command, firing the same webhook shape a real integration would produce.

In [5]:
payment_base_py = '''"""
Payment provider abstraction. Business logic (guardrail engine, order service)
only ever talks to this interface — never to Razorpay or mock internals directly.
"""
from abc import ABC, abstractmethod
from typing import Optional


class PaymentProvider(ABC):
    name: str = "base"

    @abstractmethod
    def create_order(self, amount_paise: int, currency: str, receipt: str) -> dict:
        """Returns {"order_id": ...}"""

    @abstractmethod
    def create_payment_link(self, order_id: str, amount_paise: int, currency: str,
                             description: str, upi_vpa: Optional[str] = None) -> dict:
        """Returns {"payment_link_id": ..., "payment_link_url": ...}"""

    @abstractmethod
    def fetch_payment_link(self, payment_link_id: str) -> dict:
        """Returns current status of a payment link."""

    @abstractmethod
    def capture_payment(self, payment_id: str, amount_paise: int) -> dict:
        """Captures an authorized payment."""

    @abstractmethod
    def fetch_payment(self, payment_id: str) -> dict:
        """Returns payment details."""

    @abstractmethod
    def create_refund(self, payment_id: str, amount_paise: int) -> dict:
        """Issues a refund."""

    @abstractmethod
    def simulate_webhook(self, payment_link_id: str, outcome: str) -> dict:
        """Demo-only helper: mock providers can synthesize a webhook payload.
        Real provider raises NotImplementedError (webhooks come from Razorpay itself)."""
'''

with open("kaapi_copilot/backend/app/providers/__init__.py", "w") as f:
    f.write("")
with open("kaapi_copilot/backend/app/providers/payment/__init__.py", "w") as f:
    f.write("")
with open("kaapi_copilot/backend/app/providers/payment/base.py", "w") as f:
    f.write(payment_base_py)

mock_payment_py = '''"""
MockPaymentProvider — deterministic, in-memory simulation of Razorpay's
order / payment-link / webhook flow. Recognizes the same demo UPI test
handles Razorpay itself documents: success@razorpay, failure@razorpay.
"""
import hashlib
import time
from typing import Optional
from app.providers.payment.base import PaymentProvider


class MockPaymentProvider(PaymentProvider):
    name = "mock"

    def __init__(self):
        self._orders = {}
        self._links = {}
        self._payments = {}
        self._counter = 0

    def _next_id(self, prefix: str) -> str:
        self._counter += 1
        raw = f"{prefix}-{self._counter}-{time.time()}"
        return f"{prefix}_{hashlib.sha1(raw.encode()).hexdigest()[:14]}"

    def create_order(self, amount_paise: int, currency: str, receipt: str) -> dict:
        order_id = self._next_id("order")
        self._orders[order_id] = {
            "order_id": order_id, "amount_paise": amount_paise,
            "currency": currency, "receipt": receipt, "status": "created",
        }
        return {"order_id": order_id, "status": "created"}

    def create_payment_link(self, order_id: str, amount_paise: int, currency: str,
                             description: str, upi_vpa: Optional[str] = None) -> dict:
        link_id = self._next_id("plink")
        url = f"https://mock-razorpay.test/pay/{link_id}"
        self._links[link_id] = {
            "payment_link_id": link_id, "order_id": order_id,
            "amount_paise": amount_paise, "currency": currency,
            "description": description, "upi_vpa": upi_vpa or "pending",
            "status": "created", "payment_id": None,
        }
        return {"payment_link_id": link_id, "payment_link_url": url, "status": "created"}

    def fetch_payment_link(self, payment_link_id: str) -> dict:
        return self._links.get(payment_link_id, {"status": "not_found"})

    def capture_payment(self, payment_id: str, amount_paise: int) -> dict:
        p = self._payments.get(payment_id)
        if not p:
            return {"status": "not_found"}
        p["status"] = "captured"
        return p

    def fetch_payment(self, payment_id: str) -> dict:
        return self._payments.get(payment_id, {"status": "not_found"})

    def create_refund(self, payment_id: str, amount_paise: int) -> dict:
        refund_id = self._next_id("rfnd")
        return {"refund_id": refund_id, "payment_id": payment_id,
                "amount_paise": amount_paise, "status": "processed"}

    def simulate_webhook(self, payment_link_id: str, outcome: str) -> dict:
        """outcome: 'success' -> payment.captured, 'failure' -> payment.failed"""
        link = self._links.get(payment_link_id)
        if not link:
            return {"status": "not_found"}
        payment_id = self._next_id("pay")
        vpa = "success@razorpay" if outcome == "success" else "failure@razorpay"
        if outcome == "success":
            link["status"] = "paid"
            self._payments[payment_id] = {
                "payment_id": payment_id, "order_id": link["order_id"],
                "amount_paise": link["amount_paise"], "status": "captured", "vpa": vpa,
            }
            event = "payment.captured"
        else:
            link["status"] = "failed"
            self._payments[payment_id] = {
                "payment_id": payment_id, "order_id": link["order_id"],
                "amount_paise": link["amount_paise"], "status": "failed", "vpa": vpa,
                "error_description": "Payment declined by UPI simulator (failure@razorpay).",
            }
            event = "payment.failed"
        link["payment_id"] = payment_id
        return {
            "event": event,
            "payload": {
                "payment": {"entity": self._payments[payment_id]},
                "order_id": link["order_id"],
                "payment_link_id": payment_link_id,
            },
        }


mock_payment_provider = MockPaymentProvider()
'''

with open("kaapi_copilot/backend/app/providers/payment/mock.py", "w") as f:
    f.write(mock_payment_py)

from app.providers.payment.mock import mock_payment_provider
o = mock_payment_provider.create_order(65000, "INR", "demo-receipt-1")
link = mock_payment_provider.create_payment_link(o["order_id"], 65000, "INR", "Test order")
print(o, link)
wh = mock_payment_provider.simulate_webhook(link["payment_link_id"], "failure")
print(wh)


{'order_id': 'order_6fd1c465474c28', 'status': 'created'} {'payment_link_id': 'plink_fc4c46127ca832', 'payment_link_url': 'https://mock-razorpay.test/pay/plink_fc4c46127ca832', 'status': 'created'}

{'event': 'payment.failed', 'payload': {'payment': {'entity': {'payment_id': 'pay_5b5fbdedb53180', 'order_id': 'order_6fd1c465474c28', 'amount_paise': 65000, 'status': 'failed', 'vpa': 'failure@razorpay', 'error_description': 'Payment declined by UPI simulator (failure@razorpay).'}}, 'order_id': 'order_6fd1c465474c28', 'payment_link_id': 'plink_fc4c46127ca832'}}


## 7. Real Razorpay Integration Layer

`RazorpayPaymentProvider` implements the same `PaymentProvider` interface using the official `razorpay` Python SDK (test-mode key/secret via Basic Auth) — `create_order`, `create_payment_link`, capture, refund, and payment fetch map directly onto Razorpay's REST API. `simulate_webhook` is intentionally not supported here (real webhooks come from Razorpay itself). A small factory picks Mock vs Razorpay based on `settings.payment_mode`, so the rest of the app never branches on provider type.

In [6]:
razorpay_provider_py = '''"""
RazorpayPaymentProvider — real integration layer using the official `razorpay`
Python SDK against test-mode keys. Implements the same PaymentProvider
interface as the mock, so guardrail/order logic is provider-agnostic.

Activate with:
    PAYMENT_MODE=razorpay
    RAZORPAY_KEY_ID=...
    RAZORPAY_KEY_SECRET=...
"""
from typing import Optional
from app.providers.payment.base import PaymentProvider
from app.core.config import settings


class RazorpayPaymentProvider(PaymentProvider):
    name = "razorpay"

    def __init__(self, key_id: str, key_secret: str):
        try:
            import razorpay
        except ImportError as e:
            raise RuntimeError(
                "razorpay SDK not installed. Run: pip install razorpay"
            ) from e
        self.client = razorpay.Client(auth=(key_id, key_secret))

    def create_order(self, amount_paise: int, currency: str, receipt: str) -> dict:
        order = self.client.order.create({
            "amount": amount_paise, "currency": currency, "receipt": receipt,
        })
        return {"order_id": order["id"], "status": order["status"]}

    def create_payment_link(self, order_id: str, amount_paise: int, currency: str,
                             description: str, upi_vpa: Optional[str] = None) -> dict:
        payload = {
            "amount": amount_paise, "currency": currency, "description": description,
            "notes": {"order_id": order_id},
        }
        link = self.client.payment_link.create(payload)
        return {"payment_link_id": link["id"], "payment_link_url": link["short_url"],
                "status": link["status"]}

    def fetch_payment_link(self, payment_link_id: str) -> dict:
        return self.client.payment_link.fetch(payment_link_id)

    def capture_payment(self, payment_id: str, amount_paise: int) -> dict:
        return self.client.payment.capture(payment_id, amount_paise)

    def fetch_payment(self, payment_id: str) -> dict:
        return self.client.payment.fetch(payment_id)

    def create_refund(self, payment_id: str, amount_paise: int) -> dict:
        return self.client.payment.refund(payment_id, {"amount": amount_paise})

    def simulate_webhook(self, payment_link_id: str, outcome: str) -> dict:
        raise NotImplementedError(
            "Real webhooks come from Razorpay itself via /api/webhooks/razorpay; "
            "use test UPI handles success@razorpay / failure@razorpay to trigger them."
        )


def verify_webhook_signature(body: bytes, signature: str, webhook_secret: str) -> bool:
    """HMAC-SHA256 verification per Razorpay docs. Always run before trusting a payload."""
    import hmac
    import hashlib
    expected = hmac.new(webhook_secret.encode(), body, hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected, signature or "")
'''

with open("kaapi_copilot/backend/app/providers/payment/razorpay_provider.py", "w") as f:
    f.write(razorpay_provider_py)

factory_py = '''"""
Payment provider factory — selects Mock or Razorpay based on settings.payment_mode.
The rest of the application only ever imports `get_payment_provider`.
"""
from app.core.config import settings
from app.providers.payment.mock import mock_payment_provider


def get_payment_provider():
    if settings.payment_mode == "razorpay":
        from app.providers.payment.razorpay_provider import RazorpayPaymentProvider
        return RazorpayPaymentProvider(settings.razorpay_key_id, settings.razorpay_key_secret)
    return mock_payment_provider
'''

with open("kaapi_copilot/backend/app/providers/payment/factory.py", "w") as f:
    f.write(factory_py)

from app.providers.payment.factory import get_payment_provider
provider = get_payment_provider()
print("Active payment provider:", provider.name)


Active payment provider: mock


## 8–9. AI-Agent Abstraction + Mock Rule-Based Shopping Agent

`ShoppingAgent` interface exposes one method: `handle_turn(session_state, user_message) -> AgentResponse` (reply text, proposed cart actions, at most one upsell suggestion, and an optional "ready to checkout" flag). `MockShoppingAgent` implements this with deterministic keyword/category matching against the catalog — no external API call — so Journey A and B are fully demoable with zero credentials. It enforces the same "at most one upsell per turn" and "never invent a product/price" rules as the real agent will.

In [7]:
agent_base_py = '''"""
Shopping agent abstraction. Both the mock (rule-based) and Anthropic-powered
implementations return the same AgentResponse shape, so the API layer and
Journey B MCP tools never branch on which brain is running.
"""
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class AgentResponse:
    reply: str
    add_to_cart_skus: list = field(default_factory=list)   # SKUs the agent proposes adding
    upsell_sku: Optional[str] = None                        # at most one, catalog-derived
    upsell_reason: str = ""
    ready_to_checkout: bool = False


class ShoppingAgent(ABC):
    name: str = "base"

    @abstractmethod
    def handle_turn(self, session_state: dict, user_message: str) -> AgentResponse:
        """session_state: {"cart_skus": [...], "upsell_offered": [...], "history": [...]}"""
'''

with open("kaapi_copilot/backend/app/providers/agent/__init__.py", "w") as f:
    f.write("")
with open("kaapi_copilot/backend/app/providers/agent/base.py", "w") as f:
    f.write(agent_base_py)

mock_agent_py = '''"""
MockShoppingAgent — deterministic, rule-based conversational brain. No external
API calls. Matches buyer intent against the catalog via keywords, proposes
exactly one catalog-defined upsell, and never invents a product or price.
"""
import re
from app.providers.agent.base import ShoppingAgent, AgentResponse
from app.services.catalog_service import catalog_service

INTENT_KEYWORDS = {
    "kr-filter-500": ["filter coffee", "filter", "powder", "south indian", "kaapi", "basic", "nothing fancy"],
    "kr-arabica-250": ["arabica", "single origin", "beans", "single-origin"],
    "kr-subscription": ["subscription", "monthly", "subscribe"],
    "kr-frother": ["frother", "milk foam", "froth"],
    "kr-filters-100": ["filter paper", "reusable filter", "paper filter"],
    "kr-dripper": ["dripper", "pour over", "pour-over"],
    "kr-steel-filter": ["steel filter", "filter set", "traditional filter"],
}

CHECKOUT_PHRASES = ["checkout", "buy now", "pay", "proceed", "place order", "i'm ready", "im ready", "confirm order"]
DECLINE_PHRASES = ["no thanks", "no thank you", "not now", "skip", "don't need", "dont need", "nah"]
ACCEPT_PHRASES = ["yes", "sure", "add it", "sounds good", "okay", "ok", "yeah", "add that"]


class MockShoppingAgent(ShoppingAgent):
    name = "mock"

    def _match_products(self, text: str) -> list:
        text = text.lower()
        matched = []
        for sku, keywords in INTENT_KEYWORDS.items():
            if any(kw in text for kw in keywords):
                matched.append(sku)
        return matched

    def handle_turn(self, session_state: dict, user_message: str) -> AgentResponse:
        text = user_message.lower().strip()
        cart_skus = session_state.get("cart_skus", [])
        upsell_offered = session_state.get("upsell_offered", [])
        pending_upsell = session_state.get("pending_upsell")

        # Buyer responding to a previously offered upsell
        if pending_upsell and any(p in text for p in ACCEPT_PHRASES):
            product = catalog_service.get_product(pending_upsell)
            return AgentResponse(
                reply=f"Great — added {product['name']} (₹{product['price_paise']/100:.0f}) to your cart. Ready to checkout whenever you are.",
                add_to_cart_skus=[pending_upsell],
                ready_to_checkout=False,
            )
        if pending_upsell and any(p in text for p in DECLINE_PHRASES):
            return AgentResponse(
                reply="No problem, keeping it simple. Let me know when you're ready to checkout.",
                ready_to_checkout=False,
            )

        # Explicit checkout intent
        if any(p in text for p in CHECKOUT_PHRASES) and cart_skus:
            return AgentResponse(
                reply="Here's your order summary — please review and tap 'Proceed to Pay' to confirm.",
                ready_to_checkout=True,
            )

        # Product discovery
        matched = self._match_products(text)
        if not matched:
            return AgentResponse(
                reply="Tell me what you're after — e.g. 'I want good filter coffee, nothing fancy', "
                      "or 'I like single-origin beans'. I can also show the full menu.",
            )

        primary_sku = matched[0]
        product = catalog_service.get_product(primary_sku)
        reply = f"I'd recommend {product['name']} — ₹{product['price_paise']/100:.0f}. {product['description']}"

        upsell = catalog_service.get_upsell_for(primary_sku)
        upsell_sku, upsell_reason = None, ""
        if upsell and upsell["sku"] not in upsell_offered and upsell["sku"] not in cart_skus:
            upsell_sku = upsell["sku"]
            upsell_reason = (f"Buyers who get {product['name']} usually pair it with "
                              f"{upsell['name']} (₹{upsell['price_paise']/100:.0f}) — want me to add it?")
            reply += f" {upsell_reason}"

        return AgentResponse(
            reply=reply,
            add_to_cart_skus=[primary_sku],
            upsell_sku=upsell_sku,
            upsell_reason=upsell_reason,
        )


mock_shopping_agent = MockShoppingAgent()
'''

with open("kaapi_copilot/backend/app/providers/agent/mock.py", "w") as f:
    f.write(mock_agent_py)

from app.providers.agent.mock import mock_shopping_agent
r = mock_shopping_agent.handle_turn({"cart_skus": [], "upsell_offered": []}, "I want good filter coffee, nothing fancy")
print(r)


AgentResponse(reply="I'd recommend Filter Coffee Powder, 500g — ₹450. Classic South Indian filter coffee blend. Buyers who get Filter Coffee Powder, 500g usually pair it with South Indian Steel Filter Set (₹650) — want me to add it?", add_to_cart_skus=['kr-filter-500'], upsell_sku='kr-steel-filter', upsell_reason='Buyers who get Filter Coffee Powder, 500g usually pair it with South Indian Steel Filter Set (₹650) — want me to add it?', ready_to_checkout=False)


## 10. Groq-Powered Shopping Agent

`GroqShoppingAgent` implements the same `ShoppingAgent` interface using the Groq Chat Completions API (OpenAI-compatible tool-calling) with tools (`search_catalog`, `add_to_cart`, `propose_upsell`, `view_cart`, `ready_to_checkout`). System prompt hard-caps it to at most one upsell per turn and forbids stating any price not returned by a tool call. A factory (`get_shopping_agent`) selects Mock vs Groq based on `settings.agent_mode` — falling back to mock automatically if `GROQ_API_KEY` is absent.

In [2]:
groq_agent_py = '''"""
GroqShoppingAgent -- real conversational brain using Groq's OpenAI-compatible
Chat Completions API with tool calling. Implements the same ShoppingAgent
interface as the mock.

System prompt hard constraints:
  - at most ONE upsell suggestion per turn, and only if catalog upsell_pairs says so
  - never state a price/product not returned by a tool call

Activate with:
    AGENT_MODE=groq
    GROQ_API_KEY=...
"""
import json
from app.providers.agent.base import ShoppingAgent, AgentResponse
from app.services.catalog_service import catalog_service
from app.core.config import settings

SYSTEM_PROMPT = """You are Kaapi Copilot, the shopping assistant for Kaapi Roasters (D2C filter coffee).
Rules you must never break:
- Only use products/prices returned by the search_catalog or add_to_cart tools. Never invent a product or price.
- Propose AT MOST ONE upsell per turn, and only a catalog-defined upsell pair (via propose_upsell tool), never a guess.
- Keep replies short and plain-language. When the buyer signals they are ready to pay, call ready_to_checkout.
"""

TOOLS = [
    {"type": "function", "function": {"name": "search_catalog", "description": "Search the product catalog by keyword.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "add_to_cart", "description": "Add a product SKU to the buyer's cart.",
     "parameters": {"type": "object", "properties": {"sku": {"type": "string"}}, "required": ["sku"]}}},
    {"type": "function", "function": {"name": "propose_upsell", "description": "Propose exactly one catalog-defined upsell for a SKU already in cart.",
     "parameters": {"type": "object", "properties": {"sku": {"type": "string"}}, "required": ["sku"]}}},
    {"type": "function", "function": {"name": "view_cart", "description": "View current cart contents.",
     "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {"name": "ready_to_checkout", "description": "Signal the buyer wants to proceed to payment.",
     "parameters": {"type": "object", "properties": {}}}},
]


class GroqShoppingAgent(ShoppingAgent):
    name = "groq"

    def __init__(self, api_key: str, model: str):
        try:
            from groq import Groq
        except ImportError as e:
            raise RuntimeError("groq SDK not installed. Run: pip install groq") from e
        self.client = Groq(api_key=api_key)
        self.model = model

    def _run_tool(self, tool_name: str, tool_input: dict) -> dict:
        if tool_name == "search_catalog":
            return {"results": catalog_service.search(tool_input.get("query", ""))}
        if tool_name == "add_to_cart":
            return {"product": catalog_service.get_product(tool_input["sku"])}
        if tool_name == "propose_upsell":
            return {"upsell": catalog_service.get_upsell_for(tool_input["sku"])}
        if tool_name == "view_cart":
            return {"ok": True}
        if tool_name == "ready_to_checkout":
            return {"ok": True}
        return {"error": f"unknown tool {tool_name}"}

    def handle_turn(self, session_state: dict, user_message: str) -> AgentResponse:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_message}]
        add_to_cart_skus, upsell_sku, upsell_reason, ready = [], None, "", False
        upsell_count = 0

        for _ in range(5):  # bounded tool loop
            resp = self.client.chat.completions.create(
                model=self.model, messages=messages, tools=TOOLS, max_tokens=1024,
            )
            msg = resp.choices[0].message
            messages.append(msg.model_dump() if hasattr(msg, "model_dump") else msg)

            tool_calls = getattr(msg, "tool_calls", None)
            if not tool_calls:
                return AgentResponse(msg.content or "", add_to_cart_skus, upsell_sku, upsell_reason, ready)

            for call in tool_calls:
                tool_name = call.function.name
                try:
                    tool_input = json.loads(call.function.arguments or "{}")
                except json.JSONDecodeError:
                    tool_input = {}
                result = self._run_tool(tool_name, tool_input)
                if tool_name == "add_to_cart" and result.get("product"):
                    add_to_cart_skus.append(tool_input["sku"])
                if tool_name == "propose_upsell" and result.get("upsell") and upsell_count < settings.max_upsells_per_turn:
                    upsell_sku = result["upsell"]["sku"]
                    upsell_reason = f"Catalog-defined pair for this purchase: {result['upsell']['name']}."
                    upsell_count += 1
                if tool_name == "ready_to_checkout":
                    ready = True
                messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

        return AgentResponse("Sorry, I'm having trouble responding right now.", add_to_cart_skus, upsell_sku, upsell_reason, ready)
'''

with open("kaapi_copilot/backend/app/providers/agent/groq_agent.py", "w") as f:
    f.write(groq_agent_py)

agent_factory_py = '''"""
Shopping agent factory -- selects Mock or Groq based on settings.agent_mode.
"""
from app.core.config import settings
from app.providers.agent.mock import mock_shopping_agent


def get_shopping_agent():
    if settings.agent_mode == "groq":
        from app.providers.agent.groq_agent import GroqShoppingAgent
        return GroqShoppingAgent(settings.groq_api_key, settings.groq_model)
    return mock_shopping_agent
'''

with open("kaapi_copilot/backend/app/providers/agent/factory.py", "w") as f:
    f.write(agent_factory_py)

import os
if os.path.exists("kaapi_copilot/backend/app/providers/agent/anthropic_agent.py"):
    os.remove("kaapi_copilot/backend/app/providers/agent/anthropic_agent.py")

import sys
for mod in list(sys.modules):
    if mod.startswith("app."):
        del sys.modules[mod]

from app.providers.agent.factory import get_shopping_agent
agent = get_shopping_agent()
print("Active shopping agent:", agent.name)


Active shopping agent: mock


## 12. Audit / Event System (Hash-Chained)

`AuditTrail` is an append-only, SQLite-backed log. Every entry stores a SHA-256 hash of itself chained to the previous entry's hash (`prev_hash`) — any tampering breaks the chain and is detectable via `verify_chain()`. Every tool call, guardrail check, mandate, Razorpay call, and webhook event gets logged here, in that order, always *before* any money-moving call is made.

In [9]:
audit_trail_py = '''"""
Append-only, hash-chained audit trail backed by SQLite. Every entry embeds a
SHA-256 hash of (prev_hash + event content) — tampering with any past row
breaks the chain, detectable via verify_chain().
"""
import hashlib
import json
import sqlite3
from datetime import datetime, timezone
from app.models.domain import AuditEvent, new_event_id
from app.core.config import settings

GENESIS_HASH = "0" * 64


class AuditTrail:
    def __init__(self, db_path: str = None):
        self.db_path = db_path or settings.db_path
        self._init_db()

    def _conn(self):
        return sqlite3.connect(self.db_path)

    def _init_db(self):
        with self._conn() as c:
            c.execute("""CREATE TABLE IF NOT EXISTS audit_log (
                event_id TEXT PRIMARY KEY, ts TEXT, event_type TEXT,
                session_id TEXT, payload TEXT, prev_hash TEXT, hash TEXT
            )""")

    def _last_hash(self, conn) -> str:
        row = conn.execute("SELECT hash FROM audit_log ORDER BY rowid DESC LIMIT 1").fetchone()
        return row[0] if row else GENESIS_HASH

    def log(self, event_type: str, session_id: str, payload: dict) -> AuditEvent:
        with self._conn() as c:
            prev_hash = self._last_hash(c)
            ts = datetime.now(timezone.utc).isoformat()
            event_id = new_event_id()
            payload_json = json.dumps(payload, sort_keys=True, default=str)
            digest_input = f"{event_id}|{ts}|{event_type}|{session_id}|{payload_json}|{prev_hash}"
            entry_hash = hashlib.sha256(digest_input.encode()).hexdigest()
            c.execute(
                "INSERT INTO audit_log VALUES (?,?,?,?,?,?,?)",
                (event_id, ts, event_type, session_id, payload_json, prev_hash, entry_hash),
            )
            return AuditEvent(event_id, ts, event_type, session_id, payload, prev_hash, entry_hash)

    def list_events(self, session_id: str = None, limit: int = 200) -> list:
        with self._conn() as c:
            if session_id:
                rows = c.execute(
                    "SELECT * FROM audit_log WHERE session_id=? ORDER BY rowid DESC LIMIT ?",
                    (session_id, limit)).fetchall()
            else:
                rows = c.execute("SELECT * FROM audit_log ORDER BY rowid DESC LIMIT ?", (limit,)).fetchall()
        events = []
        for r in rows:
            events.append({
                "event_id": r[0], "ts": r[1], "event_type": r[2], "session_id": r[3],
                "payload": json.loads(r[4]), "prev_hash": r[5], "hash": r[6],
            })
        return events

    def verify_chain(self) -> dict:
        """Recomputes every hash in insertion order; reports first break, if any."""
        with self._conn() as c:
            rows = c.execute("SELECT * FROM audit_log ORDER BY rowid ASC").fetchall()
        prev_hash = GENESIS_HASH
        for i, r in enumerate(rows):
            event_id, ts, event_type, session_id, payload_json, stored_prev, stored_hash = r
            if stored_prev != prev_hash:
                return {"valid": False, "broken_at_row": i, "event_id": event_id, "reason": "prev_hash mismatch"}
            digest_input = f"{event_id}|{ts}|{event_type}|{session_id}|{payload_json}|{stored_prev}"
            recomputed = hashlib.sha256(digest_input.encode()).hexdigest()
            if recomputed != stored_hash:
                return {"valid": False, "broken_at_row": i, "event_id": event_id, "reason": "hash mismatch (tampered)"}
            prev_hash = stored_hash
        return {"valid": True, "entries": len(rows)}


audit_trail = AuditTrail()
'''

with open("kaapi_copilot/backend/app/services/audit_trail.py", "w") as f:
    f.write(audit_trail_py)

import os
if os.path.exists("kaapi_copilot.db"):
    os.remove("kaapi_copilot.db")

from app.services.audit_trail import AuditTrail
at = AuditTrail(db_path="kaapi_copilot.db")
at.log("session_started", "sess_demo1", {"buyer_ref": "buyer_1"})
at.log("guardrail_check", "sess_demo1", {"rule": "session_spend_cap", "status": "pass"})
print(at.verify_chain())
print(at.list_events("sess_demo1"))


{'valid': True, 'entries': 2}

[{'event_id': 'evt_6d281946e9eb', 'ts': '2026-08-26T03:45:54.299499+00:00', 'event_type': 'guardrail_check', 'session_id': 'sess_demo1', 'payload': {'rule': 'session_spend_cap', 'status': 'pass'}, 'prev_hash': '03ed962fd1d87e81a2d3229263ba63064258a24f84322ba788e90ce30be71a04', 'hash': 'c98120732d1cd96e4aa9b3f1fb90307759c9372acc30147d6dbe9549911a0e47'}, {'event_id': 'evt_359eda91b7b5', 'ts': '2026-08-26T03:45:54.181815+00:00', 'event_type': 'session_started', 'session_id': 'sess_demo1', 'payload': {'buyer_ref': 'buyer_1'}, 'prev_hash': '0000000000000000000000000000000000000000000000000000000000000000', 'hash': '03ed962fd1d87e81a2d3229263ba63064258a24f84322ba788e90ce30be71a04'}]


## 11. Guardrail / Mandate Engine

Plain code — not a prompt — that decides whether a proposed cart may become a payment. `MandateEngine.build_mandate()`:
- re-reads every line-item price from `catalog_service` (never trusts agent-stated prices — mismatch is a hard block, not a silent correction)
- runs `session_spend_cap`, `transaction_spend_cap`, and `category_allowlist` policy checks
- writes the resulting mandate (`pending`/`blocked`) to the audit trail **before** returning
- exposes `confirm_mandate()` (buyer tap / MCP `confirm_and_pay`) which is the *only* path that can flip a mandate to `confirmed` — and `create_order`/`create_payment_link` are only ever reachable from a `confirmed` mandate.

In [10]:
mandate_engine_py = '''"""
Guardrail / Mandate Engine — the hard-gate layer between agent output and any
money-moving call. Plain code, not an LLM instruction: the agent proposes a
cart, this module decides whether it becomes a payment.

Hard constraints implemented here:
  - every line price is re-read from the catalog at mandate-build time
  - a numeric spend cap (session + per-transaction) is checked before any
    Razorpay call; a breach is a hard stop -> blocked mandate, never a soft warning
  - every mandate is written to the audit log BEFORE any Razorpay call
  - only a `confirmed` mandate may trigger create_order / create_payment_link
"""
from typing import Optional
from app.core.config import settings
from app.services.catalog_service import catalog_service
from app.services.audit_trail import audit_trail
from app.models.domain import (
    PurchaseMandate, PolicyCheck, Confirmation, CartItem,
    new_mandate_id,
)

ALLOWED_CATEGORIES = {"powder", "beans", "brew-gear", "subscription", "accessory", "consumable"}


class GuardrailError(Exception):
    pass


class MandateEngine:
    def __init__(self):
        self._session_spend_paise = {}  # session_id -> cumulative CONFIRMED/paid spend

    def _record_session_spend(self, session_id: str, amount_paise: int):
        self._session_spend_paise[session_id] = self._session_spend_paise.get(session_id, 0) + amount_paise

    def get_session_spend(self, session_id: str) -> int:
        return self._session_spend_paise.get(session_id, 0)

    def build_mandate(self, session_id: str, buyer_ref: str, cart_skus: list,
                       rationale: str, agent_stated_prices: Optional[dict] = None) -> PurchaseMandate:
        agent_stated_prices = agent_stated_prices or {}
        items, price_mismatch = [], None

        for sku in cart_skus:
            catalog_price = catalog_service.get_price_paise(sku)
            product = catalog_service.get_product(sku)
            if catalog_price is None or product is None:
                price_mismatch = f"SKU '{sku}' not found in catalog"
                break
            stated = agent_stated_prices.get(sku)
            if stated is not None and stated != catalog_price:
                price_mismatch = f"Agent stated {stated} paise for {sku}, catalog says {catalog_price} paise"
                break
            items.append(CartItem(sku=sku, name=product["name"], qty=1, unit_price_paise=catalog_price))

        total_paise = sum(i.subtotal_paise for i in items)
        checks = []

        if price_mismatch:
            checks.append(PolicyCheck("price_matches_catalog", "fail", detail=price_mismatch))
        else:
            checks.append(PolicyCheck("price_matches_catalog", "pass"))

        checks.append(PolicyCheck(
            "category_allowlist",
            "pass" if all(catalog_service.get_product(i.sku)["category"] in ALLOWED_CATEGORIES for i in items) else "fail",
        ))

        tx_status = "pass" if total_paise <= settings.transaction_spend_cap_paise else "fail"
        checks.append(PolicyCheck("transaction_spend_cap", tx_status, settings.transaction_spend_cap_paise,
                                   detail=f"total_paise={total_paise}"))

        projected_session_spend = self.get_session_spend(session_id) + total_paise
        sess_status = "pass" if projected_session_spend <= settings.session_spend_cap_paise else "fail"
        checks.append(PolicyCheck("session_spend_cap", sess_status, settings.session_spend_cap_paise,
                                   detail=f"projected_session_spend_paise={projected_session_spend}"))

        blocked = any(c.status == "fail" for c in checks)
        status = "blocked" if blocked else "pending"
        block_reason = "; ".join(c.detail for c in checks if c.status == "fail") if blocked else None

        mandate = PurchaseMandate(
            mandate_id=new_mandate_id(), session_id=session_id, buyer_ref=buyer_ref,
            items=items, currency="INR", total_paise=total_paise, rationale=rationale,
            policy_checks=checks, status=status, block_reason=block_reason,
        )

        # Always written to the audit trail before any Razorpay call is even considered.
        audit_trail.log("mandate_built", session_id, mandate.to_dict())
        return mandate

    def confirm_mandate(self, mandate: PurchaseMandate, method: str) -> PurchaseMandate:
        """The only path that may flip a mandate to 'confirmed'. method: 'buyer_tap' | 'mcp_confirm_and_pay'."""
        if mandate.status == "blocked":
            audit_trail.log("mandate_confirmation_rejected", mandate.session_id,
                             {"mandate_id": mandate.mandate_id, "reason": "mandate is blocked"})
            raise GuardrailError(f"Mandate {mandate.mandate_id} is blocked: {mandate.block_reason}")
        mandate.confirmation = Confirmation(method=method, status="confirmed",
                                             confirmed_at=__import__("datetime").datetime.now(
                                                 __import__("datetime").timezone.utc).isoformat())
        mandate.status = "confirmed"
        self._record_session_spend(mandate.session_id, mandate.total_paise)
        audit_trail.log("mandate_confirmed", mandate.session_id,
                         {"mandate_id": mandate.mandate_id, "method": method, "total_paise": mandate.total_paise})
        return mandate


mandate_engine = MandateEngine()
'''

with open("kaapi_copilot/backend/app/services/mandate_engine.py", "w") as f:
    f.write(mandate_engine_py)

from app.services.mandate_engine import mandate_engine, GuardrailError

m = mandate_engine.build_mandate("sess_demo1", "buyer_1", ["kr-filter-500", "kr-steel-filter"],
                                  "Buyer asked for filter coffee; steel filter accepted as upsell.")
print(m.to_dict())


{'mandate_id': 'mnd_241a845a718d', 'session_id': 'sess_demo1', 'created_at': '2026-08-26T03:46:37.006808+00:00', 'buyer_ref': 'buyer_1', 'items': [{'sku': 'kr-filter-500', 'name': 'Filter Coffee Powder, 500g', 'qty': 1, 'unit_price_paise': 45000, 'subtotal_paise': 45000}, {'sku': 'kr-steel-filter', 'name': 'South Indian Steel Filter Set', 'qty': 1, 'unit_price_paise': 65000, 'subtotal_paise': 65000}], 'currency': 'INR', 'total_paise': 110000, 'rationale': 'Buyer asked for filter coffee; steel filter accepted as upsell.', 'policy_checks': [{'rule': 'price_matches_catalog', 'status': 'pass', 'limit_paise': None, 'detail': ''}, {'rule': 'category_allowlist', 'status': 'pass', 'limit_paise': None, 'detail': ''}, {'rule': 'transaction_spend_cap', 'status': 'pass', 'limit_paise': 300000, 'detail': 'total_paise=110000'}, {'rule': 'session_spend_cap', 'status': 'pass', 'limit_paise': 500000, 'detail': 'projected_session_spend_paise=110000'}], 'confirmation': {'method': 'pending', 'status': 'pe

## 13–15. Order Service, Webhook Handling & Merchant/Buyer Journeys

`OrderService.checkout()` is the only code path allowed to touch the payment provider: it requires a `confirmed` mandate, creates an order + payment link via `get_payment_provider()`, persists an `Order`, and logs the call to the audit trail before returning. `handle_webhook_event()` processes `payment.captured` / `payment.failed` (verifying signatures when running against real Razorpay), flips order/mandate status accordingly, keeps the cart alive for `cart_hold_minutes` on failure, and logs every step — this is the Section 7 graceful-failure path.

In [ ]:
order_service_py = '''"""
Order Service — the ONLY code path allowed to call the payment provider.
Requires a confirmed mandate; no route from raw agent/LLM output reaches
create_order / create_payment_link directly.
"""
from datetime import datetime, timedelta, timezone
from typing import Optional
from app.core.config import settings
from app.providers.payment.factory import get_payment_provider
from app.services.audit_trail import audit_trail
from app.services.mandate_engine import mandate_engine, GuardrailError
from app.models.domain import Order, PurchaseMandate, new_order_id


class OrderService:
    def __init__(self):
        self._orders = {}
        self._mandates = {}
        self._link_to_order = {}
        self._held_carts = {}

    def register_mandate(self, mandate: PurchaseMandate):
        self._mandates[mandate.mandate_id] = mandate

    def get_mandate(self, mandate_id: str) -> Optional[PurchaseMandate]:
        return self._mandates.get(mandate_id)

    def checkout(self, mandate: PurchaseMandate) -> Order:
        """Only a CONFIRMED mandate may reach here."""
        if mandate.status != "confirmed":
            audit_trail.log("checkout_rejected", mandate.session_id,
                             {"mandate_id": mandate.mandate_id, "reason": f"mandate status is '{mandate.status}', not 'confirmed'"})
            raise GuardrailError(f"Cannot checkout: mandate status is '{mandate.status}', not 'confirmed'")

        provider = get_payment_provider()
        order_resp = provider.create_order(mandate.total_paise, mandate.currency, receipt=mandate.mandate_id)
        link_resp = provider.create_payment_link(
            order_resp["order_id"], mandate.total_paise, mandate.currency,
            description=f"Kaapi Roasters order for {mandate.buyer_ref}",
        )

        order = Order(
            order_id=order_resp["order_id"], mandate_id=mandate.mandate_id, session_id=mandate.session_id,
            total_paise=mandate.total_paise, currency=mandate.currency, status="created",
            payment_link_id=link_resp["payment_link_id"], payment_link_url=link_resp["payment_link_url"],
        )
        mandate.order_id = order.order_id
        self._orders[order.order_id] = order
        self._link_to_order[link_resp["payment_link_id"]] = order.order_id

        audit_trail.log("order_created", mandate.session_id, {
            "order_id": order.order_id, "mandate_id": mandate.mandate_id,
            "payment_link_id": link_resp["payment_link_id"], "payment_link_url": link_resp["payment_link_url"],
            "total_paise": order.total_paise, "provider": provider.name,
        })
        return order

    def get_order(self, order_id: str) -> Optional[Order]:
        return self._orders.get(order_id)

    def handle_webhook_event(self, event: str, payload: dict) -> Order:
        """Processes payment.captured / payment.failed. Never marks paid except from a verified webhook."""
        link_id = payload.get("payment_link_id")
        order_id = self._link_to_order.get(link_id) or payload.get("order_id")
        order = self._orders.get(order_id)
        if not order:
            audit_trail.log("webhook_order_not_found", "unknown", {"event": event, "payload": payload})
            raise ValueError(f"No order found for webhook payload: {payload}")

        if event == "payment.captured":
            order.status = "paid"
            order.updated_at = datetime.now(timezone.utc).isoformat()
            audit_trail.log("payment_captured", order.session_id,
                             {"order_id": order.order_id, "payment": payload.get("payment", {})})
        elif event == "payment.failed":
            order.status = "payment_failed"
            order.updated_at = datetime.now(timezone.utc).isoformat()
            order.failure_reason = payload.get("payment", {}).get("entity", {}).get(
                "error_description", "Payment declined.")
            held_until = (datetime.now(timezone.utc) + timedelta(minutes=settings.cart_hold_minutes)).isoformat()
            mandate = self._mandates.get(order.mandate_id)
            cart_skus = [i.sku for i in mandate.items] if mandate else []
            self._held_carts[order.session_id] = {"skus": cart_skus, "held_until": held_until}
            audit_trail.log("payment_failed", order.session_id, {
                "order_id": order.order_id, "reason": order.failure_reason,
                "cart_held_until": held_until, "recovery_offer": "retry_or_alternate_payment_method",
            })
        else:
            audit_trail.log("webhook_unhandled_event", order.session_id, {"event": event, "payload": payload})
        return order

    def get_held_cart(self, session_id: str) -> Optional[dict]:
        return self._held_carts.get(session_id)


order_service = OrderService()
'''

with open("kaapi_copilot/backend/app/services/order_service.py", "w") as f:
    f.write(order_service_py)

from app.services.order_service import order_service
from app.services.mandate_engine import mandate_engine
from app.providers.payment.mock import mock_payment_provider

m2 = mandate_engine.build_mandate("sess_demo2", "buyer_2", ["kr-arabica-250"], "Buyer wants single-origin beans.")
order_service.register_mandate(m2)
mandate_engine.confirm_mandate(m2, method="buyer_tap")
order = order_service.checkout(m2)
print("Order created:", order.to_dict())

wh = mock_payment_provider.simulate_webhook(order.payment_link_id, "failure")
updated = order_service.handle_webhook_event(wh["event"], wh["payload"])
print("Order after failed webhook:", updated.to_dict())
print("Held cart:", order_service.get_held_cart("sess_demo2"))

Order created: {'order_id': 'order_9436cb52d3975f', 'mandate_id': 'mnd_1da2a53fc25e', 'session_id': 'sess_demo2', 'total_paise': 38000, 'currency': 'INR', 'status': 'created', 'payment_link_id': 'plink_6df6267567b130', 'payment_link_url': 'https://mock-razorpay.test/pay/plink_6df6267567b130', 'created_at': '2026-08-26T03:47:21.688383+00:00', 'updated_at': '2026-08-26T03:47:21.688390+00:00', 'failure_reason': None}

Order after failed webhook: {'order_id': 'order_9436cb52d3975f', 'mandate_id': 'mnd_1da2a53fc25e', 'session_id': 'sess_demo2', 'total_paise': 38000, 'currency': 'INR', 'status': 'payment_failed', 'payment_link_id': 'plink_6df6267567b130', 'payment_link_url': 'https://mock-razorpay.test/pay/plink_6df6267567b130', 'created_at': '2026-08-26T03:47:21.688383+00:00', 'updated_at': '2026-08-26T03:47:21.798891+00:00', 'failure_reason': 'Payment declined by UPI simulator (failure@razorpay).'}

Held cart: {'skus': ['kr-arabica-250'], 'held_until': '2026-08-26T03:57:21.798897+00:00'}


## 16. Mandate / Recurring-Payment Simulation

`RecurringMandateService` demonstrates the `kr-subscription` SKU as a recurring mandate: a `RecurringMandate` is built once (bounded by the same spend caps), confirmed once, then `run_cycle()` can be invoked repeatedly (e.g. monthly) to create a fresh order/payment-link per cycle under the same mandate authorization — mirroring AP2's Intent→Cart→Payment mandate chain concept for subscriptions, fully logged to the audit trail each cycle.

In [12]:
recurring_py = '''"""
Recurring mandate simulation for subscription SKUs (e.g. kr-subscription).
A mandate is authorized once; run_cycle() re-uses that authorization to
create a fresh order/payment-link per billing cycle, gated by the same
guardrail checks each time (re-verifies price + spend caps per cycle).
"""
from datetime import datetime, timezone
from app.services.mandate_engine import mandate_engine, GuardrailError
from app.services.order_service import order_service
from app.services.audit_trail import audit_trail


class RecurringMandateService:
    def __init__(self):
        self._recurring = {}  # mandate_id -> {"session_id":..., "buyer_ref":..., "skus":[...], "cycles_run": int}

    def authorize(self, session_id: str, buyer_ref: str, skus: list) -> dict:
        mandate = mandate_engine.build_mandate(
            session_id, buyer_ref, skus,
            rationale="Buyer authorized a recurring monthly subscription mandate.",
        )
        order_service.register_mandate(mandate)
        mandate_engine.confirm_mandate(mandate, method="buyer_tap")
        self._recurring[mandate.mandate_id] = {
            "session_id": session_id, "buyer_ref": buyer_ref, "skus": skus, "cycles_run": 0,
        }
        audit_trail.log("recurring_mandate_authorized", session_id,
                         {"mandate_id": mandate.mandate_id, "skus": skus})
        return {"mandate_id": mandate.mandate_id, "status": mandate.status}

    def run_cycle(self, mandate_id: str) -> dict:
        """Re-verifies price/caps and creates a new order+link for this billing cycle."""
        info = self._recurring.get(mandate_id)
        if not info:
            raise GuardrailError(f"No recurring authorization found for {mandate_id}")

        cycle_mandate = mandate_engine.build_mandate(
            info["session_id"], info["buyer_ref"], info["skus"],
            rationale=f"Recurring billing cycle #{info['cycles_run'] + 1} under mandate {mandate_id}.",
        )
        order_service.register_mandate(cycle_mandate)
        mandate_engine.confirm_mandate(cycle_mandate, method="mcp_confirm_and_pay")
        order = order_service.checkout(cycle_mandate)
        info["cycles_run"] += 1
        audit_trail.log("recurring_cycle_billed", info["session_id"], {
            "parent_mandate_id": mandate_id, "cycle_mandate_id": cycle_mandate.mandate_id,
            "order_id": order.order_id, "cycle_number": info["cycles_run"],
        })
        return {"cycle_number": info["cycles_run"], "order": order.to_dict()}


recurring_mandate_service = RecurringMandateService()
'''

with open("kaapi_copilot/backend/app/services/recurring_mandate.py", "w") as f:
    f.write(recurring_py)

from app.services.recurring_mandate import recurring_mandate_service

auth = recurring_mandate_service.authorize("sess_sub1", "buyer_3", ["kr-subscription"])
print("Authorized:", auth)
cycle1 = recurring_mandate_service.run_cycle(auth["mandate_id"])
print("Cycle 1:", cycle1["cycle_number"], cycle1["order"]["order_id"], cycle1["order"]["total_paise"])


Authorized: {'mandate_id': 'mnd_1193e0531ae3', 'status': 'confirmed'}
Cycle 1: 1 order_f8add57ff96753 70000


## 13b. Session Manager & Revenue Analytics

`SessionManager` tracks per-session cart state (SKUs, pending upsell, offered upsells) driving Journey A's conversational turns. `AnalyticsService` derives the Section 8 revenue numbers directly from `order_service` + `mandate_engine` state: baseline vs. agent-assisted AOV, upsell-attach rate (accepted upsell / offered upsell), and successful vs. blocked vs. failed transaction counts — no separate data pipeline needed since everything is already logged.

In [13]:
session_manager_py = '''"""
In-memory session state for Journey A conversational turns. Tracks the cart,
which upsell (if any) is currently pending a buyer response, and which
upsells have already been offered (so we never re-offer the same one).
"""
from app.models.domain import new_session_id


class SessionManager:
    def __init__(self):
        self._sessions = {}  # session_id -> state dict

    def create_session(self, buyer_ref: str) -> str:
        session_id = new_session_id()
        self._sessions[session_id] = {
            "buyer_ref": buyer_ref, "cart_skus": [], "upsell_offered": [],
            "pending_upsell": None, "history": [], "upsells_accepted": 0, "upsells_offered_count": 0,
        }
        return session_id

    def get_state(self, session_id: str) -> dict:
        if session_id not in self._sessions:
            raise KeyError(f"Unknown session {session_id}")
        return self._sessions[session_id]

    def apply_agent_response(self, session_id: str, agent_response) -> None:
        state = self.get_state(session_id)
        for sku in agent_response.add_to_cart_skus:
            if sku not in state["cart_skus"]:
                state["cart_skus"].append(sku)
            if state.get("pending_upsell") == sku:
                state["upsells_accepted"] += 1
        if agent_response.upsell_sku:
            state["pending_upsell"] = agent_response.upsell_sku
            state["upsell_offered"].append(agent_response.upsell_sku)
            state["upsells_offered_count"] += 1
        elif not agent_response.add_to_cart_skus or state.get("pending_upsell") not in agent_response.add_to_cart_skus:
            state["pending_upsell"] = None


session_manager = SessionManager()
'''

with open("kaapi_copilot/backend/app/services/session_manager.py", "w") as f:
    f.write(session_manager_py)

analytics_py = '''"""
Revenue / growth analytics derived directly from order_service + mandate_engine
+ session_manager state (no separate data pipeline required).
"""
from app.services.order_service import order_service
from app.services.session_manager import session_manager

BASELINE_AOV_PAISE = 45000  # assumption: a non-agentic storefront sells one core item, no upsell


class AnalyticsService:
    def summary(self) -> dict:
        orders = list(order_service._orders.values())
        paid = [o for o in orders if o.status == "paid"]
        failed = [o for o in orders if o.status == "payment_failed"]
        created = [o for o in orders if o.status == "created"]

        mandates = list(order_service._mandates.values())
        blocked = [m for m in mandates if m.status == "blocked"]

        sessions = session_manager._sessions
        total_offered = sum(s["upsells_offered_count"] for s in sessions.values())
        total_accepted = sum(s["upsells_accepted"] for s in sessions.values())
        attach_rate = (total_accepted / total_offered) if total_offered else 0.0

        agent_assisted_aov = (sum(o.total_paise for o in paid) / len(paid)) if paid else 0

        return {
            "orders_paid": len(paid),
            "orders_payment_failed": len(failed),
            "orders_created_awaiting_payment": len(created),
            "mandates_blocked": len(blocked),
            "upsell_attach_rate_pct": round(attach_rate * 100, 1),
            "baseline_aov_paise": BASELINE_AOV_PAISE,
            "agent_assisted_aov_paise": round(agent_assisted_aov),
            "aov_lift_pct": round(((agent_assisted_aov - BASELINE_AOV_PAISE) / BASELINE_AOV_PAISE) * 100, 1)
                            if agent_assisted_aov else 0.0,
        }


analytics_service = AnalyticsService()
'''

with open("kaapi_copilot/backend/app/services/analytics_service.py", "w") as f:
    f.write(analytics_py)

from app.services.session_manager import session_manager
from app.services.analytics_service import analytics_service

sid = session_manager.create_session("buyer_demo")
from app.providers.agent.factory import get_shopping_agent
agent = get_shopping_agent()
r1 = agent.handle_turn(session_manager.get_state(sid), "I want good filter coffee, nothing fancy")
session_manager.apply_agent_response(sid, r1)
r2 = agent.handle_turn(session_manager.get_state(sid), "sure, add it")
session_manager.apply_agent_response(sid, r2)
print(session_manager.get_state(sid))
print(analytics_service.summary())


{'buyer_ref': 'buyer_demo', 'cart_skus': ['kr-filter-500', 'kr-steel-filter'], 'upsell_offered': ['kr-steel-filter'], 'pending_upsell': 'kr-steel-filter', 'history': [], 'upsells_accepted': 1, 'upsells_offered_count': 1}

{'orders_paid': 0, 'orders_payment_failed': 1, 'orders_created_awaiting_payment': 1, 'mandates_blocked': 0, 'upsell_attach_rate_pct': 100.0, 'baseline_aov_paise': 45000, 'agent_assisted_aov_paise': 0, 'aov_lift_pct': 0.0}


## 17. FastAPI Backend — REST API + MCP-style Tool Endpoints

Wires every service into a FastAPI app: Journey A chat endpoint (`/api/chat`), mandate build/confirm (`/api/mandates/*`), checkout (`/api/checkout`), the Razorpay webhook (`/api/webhooks/razorpay`) with signature verification, demo webhook triggers, audit trail (`/api/audit`), analytics (`/api/analytics`), and the Journey B MCP-style tool surface (`/api/mcp/*`: `list_products`, `get_product`, `add_to_cart`, `get_cart`, `create_checkout_mandate`, `confirm_and_pay`) — all backed by the exact same guardrail/order/audit services as Journey A.

In [ ]:
main_api_py = '''"""
Kaapi Copilot FastAPI backend. One app, all endpoints:
  Journey A:  /api/chat, /api/mandates/*, /api/checkout, /api/webhooks/razorpay
  Ops:        /api/audit, /api/analytics
  Journey B:  /api/mcp/* (MCP-style tool surface for an external AI buyer agent)

Run with: uvicorn app.api.main:app --reload --port 8000
"""
from typing import Optional
from fastapi import FastAPI, Request, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from app.core.config import settings
from app.services.catalog_service import catalog_service
from app.services.session_manager import session_manager
from app.services.mandate_engine import mandate_engine, GuardrailError
from app.services.order_service import order_service
from app.services.audit_trail import audit_trail
from app.services.analytics_service import analytics_service
from app.providers.agent.factory import get_shopping_agent
from app.providers.payment.factory import get_payment_provider
from app.providers.payment.mock import mock_payment_provider

app = FastAPI(title="Kaapi Copilot")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

# In-memory cart store for the MCP (Journey B) tool surface, keyed by mcp session id
_mcp_carts: dict = {}


class ChatRequest(BaseModel):
    session_id: Optional[str] = None
    buyer_ref: str = "buyer_web"
    message: str


class ConfirmRequest(BaseModel):
    mandate_id: str
    method: str = "buyer_tap"


class McpCartRequest(BaseModel):
    session_id: str
    sku: str


class McpMandateRequest(BaseModel):
    session_id: str
    buyer_ref: str


class McpConfirmRequest(BaseModel):
    mandate_id: str


@app.get("/api/health")
def health():
    return {"status": "ok", **settings.summary()}


# ---------------- Journey A: conversational chat ----------------
@app.post("/api/chat")
def chat(req: ChatRequest):
    session_id = req.session_id or session_manager.create_session(req.buyer_ref)
    state = session_manager.get_state(session_id)
    audit_trail.log("chat_message_received", session_id, {"buyer_ref": req.buyer_ref, "message": req.message})

    agent = get_shopping_agent()
    response = agent.handle_turn(state, req.message)
    session_manager.apply_agent_response(session_id, response)
    audit_trail.log("agent_turn", session_id, {
        "reply": response.reply, "added": response.add_to_cart_skus,
        "upsell_sku": response.upsell_sku, "ready_to_checkout": response.ready_to_checkout,
    })

    return {
        "session_id": session_id, "reply": response.reply,
        "cart_skus": state["cart_skus"], "upsell_sku": response.upsell_sku,
        "upsell_reason": response.upsell_reason, "ready_to_checkout": response.ready_to_checkout,
    }


@app.post("/api/mandates/build")
def build_mandate(req: ChatRequest):
    if not req.session_id:
        raise HTTPException(400, "session_id required")
    state = session_manager.get_state(req.session_id)
    mandate = mandate_engine.build_mandate(
        req.session_id, req.buyer_ref, state["cart_skus"],
        rationale=f"Buyer-confirmed cart built via Journey A chat for {req.buyer_ref}.",
    )
    order_service.register_mandate(mandate)
    return mandate.to_dict()


@app.post("/api/mandates/confirm")
def confirm_mandate(req: ConfirmRequest):
    mandate = order_service.get_mandate(req.mandate_id)
    if not mandate:
        raise HTTPException(404, "mandate not found")
    try:
        mandate = mandate_engine.confirm_mandate(mandate, method=req.method)
    except GuardrailError as e:
        raise HTTPException(400, str(e))
    return mandate.to_dict()


@app.post("/api/checkout")
def checkout(req: ConfirmRequest):
    mandate = order_service.get_mandate(req.mandate_id)
    if not mandate:
        raise HTTPException(404, "mandate not found")
    try:
        order = order_service.checkout(mandate)
    except GuardrailError as e:
        raise HTTPException(400, str(e))
    return order.to_dict()


# ---------------- Webhooks ----------------
@app.post("/api/webhooks/razorpay")
async def razorpay_webhook(request: Request):
    body = await request.body()
    signature = request.headers.get("X-Razorpay-Signature", "")
    if settings.payment_mode == "razorpay":
        from app.providers.payment.razorpay_provider import verify_webhook_signature
        if not verify_webhook_signature(body, signature, settings.razorpay_webhook_secret):
            audit_trail.log("webhook_signature_invalid", "unknown", {"signature": signature})
            raise HTTPException(400, "invalid signature")
    payload = await request.json()
    order = order_service.handle_webhook_event(payload["event"], payload["payload"])
    return {"status": "processed", "order_id": order.order_id, "order_status": order.status}


@app.post("/api/demo/trigger-webhook")
def trigger_demo_webhook(order_id: str, outcome: str = "success"):
    """Demo-only: simulate a Razorpay webhook without a real payment."""
    order = order_service.get_order(order_id)
    if not order:
        raise HTTPException(404, "order not found")
    if order.payment_link_id is None:
        raise ValueError("Order has no payment link")
    wh = mock_payment_provider.simulate_webhook(order.payment_link_id, outcome)
    updated = order_service.handle_webhook_event(wh["event"], wh["payload"])
    return updated.to_dict()


# ---------------- Ops: audit + analytics ----------------
@app.get("/api/audit")
def get_audit(session_id: Optional[str] = None, limit: int = 200):
    return {"events": audit_trail.list_events(session_id, limit), "chain": audit_trail.verify_chain()}


@app.get("/api/analytics")
def get_analytics():
    return analytics_service.summary()


# ---------------- Journey B: MCP-style tool surface ----------------
@app.get("/api/mcp/list_products")
def mcp_list_products(category: Optional[str] = None):
    return {"products": catalog_service.list_products(category)}


@app.get("/api/mcp/get_product")
def mcp_get_product(sku: str):
    product = catalog_service.get_product(sku)
    if not product:
        raise HTTPException(404, "sku not found")
    return product


@app.post("/api/mcp/add_to_cart")
def mcp_add_to_cart(req: McpCartRequest):
    if catalog_service.get_product(req.sku) is None:
        raise HTTPException(404, "sku not found")
    cart = _mcp_carts.setdefault(req.session_id, [])
    if req.sku not in cart:
        cart.append(req.sku)
    audit_trail.log("mcp_add_to_cart", req.session_id, {"sku": req.sku})
    return {"cart_skus": cart}


@app.get("/api/mcp/get_cart")
def mcp_get_cart(session_id: str):
    cart = _mcp_carts.get(session_id, [])
    items = [catalog_service.get_product(s) for s in cart]
    total = sum(i["price_paise"] for i in items)
    return {"cart_skus": cart, "items": items, "total_paise": total}


@app.post("/api/mcp/create_checkout_mandate")
def mcp_create_checkout_mandate(req: McpMandateRequest):
    cart = _mcp_carts.get(req.session_id, [])
    if not cart:
        raise HTTPException(400, "cart is empty")
    mandate = mandate_engine.build_mandate(
        req.session_id, req.buyer_ref, cart,
        rationale="External AI buyer agent (Journey B) requested checkout via MCP tools.",
    )
    order_service.register_mandate(mandate)
    return mandate.to_dict()


@app.post("/api/mcp/confirm_and_pay")
def mcp_confirm_and_pay(req: McpConfirmRequest):
    """The MCP equivalent of the buyer's tap — explicit confirmation with no human present."""
    mandate = order_service.get_mandate(req.mandate_id)
    if not mandate:
        raise HTTPException(404, "mandate not found")
    try:
        mandate = mandate_engine.confirm_mandate(mandate, method="mcp_confirm_and_pay")
        order = order_service.checkout(mandate)
    except GuardrailError as e:
        raise HTTPException(400, str(e))
    return {"mandate": mandate.to_dict(), "order": order.to_dict()}
'''

with open("kaapi_copilot/backend/app/api/__init__.py", "w") as f:
    f.write("")
with open("kaapi_copilot/backend/app/api/main.py", "w") as f:
    f.write(main_api_py)

with open("kaapi_copilot/backend/requirements.txt", "w") as f:
    f.write("fastapi\nuvicorn[standard]\npydantic\nrazorpay\ngroq\n")

print("FastAPI app written to kaapi_copilot/backend/app/api/main.py")
print("Run locally with: uvicorn app.api.main:app --reload --port 8000 (from kaapi_copilot/backend)")


FastAPI app written to kaapi_copilot/backend/app/api/main.py
Run locally with: uvicorn app.api.main:app --reload --port 8000 (from kaapi_copilot/backend)


In [ ]:
import sys

# Dependencies are installed from kaapi_copilot/backend/requirements.txt by the deployment environment.

# Fresh import of the FastAPI app module for a clean smoke test
for mod in list(sys.modules):
    if mod.startswith("app."):
        del sys.modules[mod]

from fastapi.testclient import TestClient
from app.api.main import app

client = TestClient(app)

print(client.get("/api/health").json())
print(client.get("/api/mcp/list_products").json()["products"][:2])

r = client.post("/api/chat", json={"buyer_ref": "buyer_smoke", "message": "I want good filter coffee, nothing fancy"})
print(r.json())
sid = r.json()["session_id"]

r2 = client.post("/api/chat", json={"session_id": sid, "buyer_ref": "buyer_smoke", "message": "sure, add it"})
print(r2.json())

r3 = client.post("/api/mandates/build", json={"session_id": sid, "buyer_ref": "buyer_smoke", "message": ""})
mandate = r3.json()
print("mandate status:", mandate["status"], "total_paise:", mandate["total_paise"])

r4 = client.post("/api/mandates/confirm", json={"mandate_id": mandate["mandate_id"]})
print("confirmed status:", r4.json()["status"])

r5 = client.post("/api/checkout", json={"mandate_id": mandate["mandate_id"]})
order = r5.json()
print("order:", order["order_id"], order["status"])

r6 = client.post(f"/api/demo/trigger-webhook?order_id={order['order_id']}&outcome=success")
print("after webhook:", r6.json()["status"])

print("analytics:", client.get("/api/analytics").json())
print("audit chain valid:", client.get("/api/audit").json()["chain"])

/opt/conda/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


{'status': 'ok', 'payment_mode_requested': 'mock', 'payment_mode_effective': 'mock', 'razorpay_keys_present': False, 'agent_mode_requested': 'mock', 'agent_mode_effective': 'mock', 'groq_key_present': False, 'session_spend_cap_paise': 500000, 'transaction_spend_cap_paise': 300000}
[{'sku': 'kr-filter-500', 'name': 'Filter Coffee Powder, 500g', 'price_paise': 45000, 'category': 'powder', 'description': 'Classic South Indian filter coffee blend.', 'upsell_pairs': ['kr-steel-filter']}, {'sku': 'kr-arabica-250', 'name': 'Single-Origin Arabica Beans, 250g', 'price_paise': 38000, 'category': 'beans', 'description': 'Light-roast single-origin arabica.', 'upsell_pairs': ['kr-dripper']}]


{'session_id': 'sess_df8b2ad933ff', 'reply': "I'd recommend Filter Coffee Powder, 500g — ₹450. Classic South Indian filter coffee blend. Buyers who get Filter Coffee Powder, 500g usually pair it with South Indian Steel Filter Set (₹650) — want me to add it?", 'cart_skus': ['kr-filter-500'], 'upsell_sku': 'kr-steel-filter', 'upsell_reason': 'Buyers who get Filter Coffee Powder, 500g usually pair it with South Indian Steel Filter Set (₹650) — want me to add it?', 'ready_to_checkout': False}


{'session_id': 'sess_df8b2ad933ff', 'reply': 'Great — added South Indian Steel Filter Set (₹650) to your cart. Ready to checkout whenever you are.', 'cart_skus': ['kr-filter-500', 'kr-steel-filter'], 'upsell_sku': None, 'upsell_reason': '', 'ready_to_checkout': False}
mandate status: pending total_paise: 110000


confirmed status: confirmed
order: order_2ba515451c7f5b created


after webhook: paid
analytics: {'orders_paid': 1, 'orders_payment_failed': 0, 'orders_created_awaiting_payment': 0, 'mandates_blocked': 0, 'upsell_attach_rate_pct': 100.0, 'baseline_aov_paise': 45000, 'agent_assisted_aov_paise': 110000, 'aov_lift_pct': 144.4}
audit chain valid: {'valid': True, 'entries': 40}


## Section 7 Failure Demos — UPI Decline & Guardrail Rejection

Two deliberate failure paths, both fully logged:
1. **UPI decline** (`failure@razorpay`): checkout → webhook `payment.failed` → order stays `payment_failed` (never `paid`), cart held for `cart_hold_minutes`, recovery offer logged.
2. **Guardrail rejection**: a cart exceeding `transaction_spend_cap_paise`/`session_spend_cap_paise` is hard-blocked at `build_mandate()` — status `blocked`, and any attempt to confirm or checkout it raises `GuardrailError`, never silently passing.

In [17]:
# --- Demo 1: UPI decline (failure@razorpay) ---
r = client.post("/api/chat", json={"buyer_ref": "buyer_fail", "message": "I like single-origin beans"})
sid2 = r.json()["session_id"]
mandate2 = client.post("/api/mandates/build", json={"session_id": sid2, "buyer_ref": "buyer_fail", "message": ""}).json()
client.post("/api/mandates/confirm", json={"mandate_id": mandate2["mandate_id"]})
order2 = client.post("/api/checkout", json={"mandate_id": mandate2["mandate_id"]}).json()
failed_order = client.post(f"/api/demo/trigger-webhook?order_id={order2['order_id']}&outcome=failure").json()
print("UPI decline -> order status:", failed_order["status"], "| reason:", failed_order["failure_reason"])

# --- Demo 2: guardrail rejection (over transaction cap) ---
r = client.post("/api/chat", json={"buyer_ref": "buyer_over_cap", "message": "I want a pour over dripper"})
sid3 = r.json()["session_id"]
# manually push cart over the ₹3,000 transaction cap by adding subscription + dripper + frother
from app.services.session_manager import session_manager
state3 = session_manager.get_state(sid3)
state3["cart_skus"] = ["kr-subscription", "kr-dripper", "kr-frother", "kr-steel-filter", "kr-arabica-250"]
mandate3 = client.post("/api/mandates/build", json={"session_id": sid3, "buyer_ref": "buyer_over_cap", "message": ""}).json()
print("Over-cap mandate status:", mandate3["status"], "| block_reason:", mandate3["block_reason"])

confirm_attempt = client.post("/api/mandates/confirm", json={"mandate_id": mandate3["mandate_id"]})
print("Confirm attempt on blocked mandate -> HTTP", confirm_attempt.status_code, confirm_attempt.json())

print("\\nFinal audit chain check:", client.get("/api/audit").json()["chain"])


UPI decline -> order status: payment_failed | reason: Payment declined by UPI simulator (failure@razorpay).
Over-cap mandate status: blocked | block_reason: total_paise=318000

Confirm attempt on blocked mandate -> HTTP 400 {'detail': 'Mandate mnd_a10601f62e92 is blocked: total_paise=318000'}

\nFinal audit chain check: {'valid': True, 'entries': 32}


## 18. Frontend / Demo UI

A single static HTML file (`frontend/index.html`, no build step) with three panels: **Journey A chat** (buyer message box, agent replies, cart, "Proceed to Pay" button gated behind a built+confirmed mandate), **Ops** (live audit trail viewer + hash-chain validity + analytics numbers), and **Journey B simulator** (buttons that call the `/api/mcp/*` tools in sequence to show an external agent shopping with no human typing). Talks to the FastAPI backend via `fetch`.

In [18]:
frontend_html = '''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Kaapi Copilot</title>
<style>
  body { font-family: -apple-system, Segoe UI, sans-serif; background:#f5f1ea; margin:0; padding:20px; color:#3a2a1a; }
  h1 { color:#6b3e1f; }
  .grid { display:grid; grid-template-columns: 1fr 1fr; gap:20px; }
  .panel { background:#fff; border-radius:10px; padding:16px; box-shadow:0 2px 6px rgba(0,0,0,.08); }
  .chat-log { height:280px; overflow-y:auto; border:1px solid #eee; padding:10px; border-radius:8px; margin-bottom:10px; background:#faf7f2; }
  .msg-buyer { text-align:right; color:#1a5c3a; margin:6px 0; }
  .msg-agent { text-align:left; color:#6b3e1f; margin:6px 0; }
  input, button, select { padding:8px; border-radius:6px; border:1px solid #ccc; font-size:14px; }
  button { background:#6b3e1f; color:#fff; border:none; cursor:pointer; }
  button:hover { background:#8a5228; }
  button:disabled { background:#ccc; cursor:not-allowed; }
  #cart, #mandateBox { font-size:13px; background:#faf7f2; padding:8px; border-radius:6px; margin-top:8px; }
  pre { font-size:11px; max-height:220px; overflow:auto; background:#111; color:#c9f7c9; padding:8px; border-radius:6px; }
  .badge { display:inline-block; padding:2px 8px; border-radius:12px; font-size:12px; color:#fff; }
  .pass { background:#2a7a3a; } .fail { background:#a53030; } .pending { background:#a5842e; }
</style>
</head>
<body>
<h1>☕ Kaapi Copilot</h1>
<div class="grid">
  <div class="panel">
    <h3>Journey A — Chat with the shopping agent</h3>
    <div class="chat-log" id="chatLog"></div>
    <input id="msgInput" placeholder="I want good filter coffee, nothing fancy" style="width:70%">
    <button onclick="sendMessage()">Send</button>
    <div id="cart">Cart: (empty)</div>
    <button id="mandateBtn" onclick="buildMandate()">Build Mandate</button>
    <div id="mandateBox"></div>
    <button id="payBtn" onclick="confirmAndPay()" disabled>Proceed to Pay</button>
    <button onclick="triggerWebhook('success')">Simulate success@razorpay</button>
    <button onclick="triggerWebhook('failure')">Simulate failure@razorpay</button>
    <div id="orderBox"></div>
  </div>

  <div class="panel">
    <h3>Ops — Audit Trail &amp; Analytics</h3>
    <button onclick="refreshAudit()">Refresh Audit Trail</button>
    <div id="chainStatus"></div>
    <pre id="auditLog"></pre>
    <h4>Revenue Analytics</h4>
    <pre id="analyticsBox"></pre>
  </div>
</div>

<div class="panel" style="margin-top:20px;">
  <h3>Journey B — External AI buyer agent (MCP tools, no human typing)</h3>
  <button onclick="runJourneyB()">Run Journey B demo (budget ₹1200)</button>
  <pre id="journeyBLog"></pre>
</div>

<script>
const API = "http://localhost:8000/api";
let sessionId = null, currentMandateId = null, currentOrderId = null;

function logChat(who, text) {
  const log = document.getElementById("chatLog");
  const div = document.createElement("div");
  div.className = who === "buyer" ? "msg-buyer" : "msg-agent";
  div.textContent = (who === "buyer" ? "You: " : "Kaapi: ") + text;
  log.appendChild(div);
  log.scrollTop = log.scrollHeight;
}

async function sendMessage() {
  const input = document.getElementById("msgInput");
  const message = input.value.trim();
  if (!message) return;
  logChat("buyer", message);
  input.value = "";
  const res = await fetch(`${API}/chat`, {
    method: "POST", headers: {"Content-Type": "application/json"},
    body: JSON.stringify({ session_id: sessionId, buyer_ref: "web_buyer", message }),
  });
  const data = await res.json();
  sessionId = data.session_id;
  logChat("agent", data.reply);
  document.getElementById("cart").textContent = "Cart: " + (data.cart_skus.join(", ") || "(empty)");
}

async function buildMandate() {
  if (!sessionId) return;
  const res = await fetch(`${API}/mandates/build`, {
    method: "POST", headers: {"Content-Type": "application/json"},
    body: JSON.stringify({ session_id: sessionId, buyer_ref: "web_buyer", message: "" }),
  });
  const mandate = await res.json();
  currentMandateId = mandate.mandate_id;
  document.getElementById("mandateBox").innerHTML =
    `Mandate ${mandate.mandate_id} — status: <span class="badge ${mandate.status === 'blocked' ? 'fail' : 'pending'}">${mandate.status}</span>` +
    `<br>Total: ₹${(mandate.total_paise/100).toFixed(0)}<br>` +
    mandate.policy_checks.map(c => `<span class="badge ${c.status === 'pass' ? 'pass' : 'fail'}">${c.rule}: ${c.status}</span>`).join(" ");
  document.getElementById("payBtn").disabled = mandate.status === "blocked";
}

async function confirmAndPay() {
  if (!currentMandateId) return;
  await fetch(`${API}/mandates/confirm`, {
    method: "POST", headers: {"Content-Type": "application/json"},
    body: JSON.stringify({ mandate_id: currentMandateId, method: "buyer_tap" }),
  });
  const res = await fetch(`${API}/checkout`, {
    method: "POST", headers: {"Content-Type": "application/json"},
    body: JSON.stringify({ mandate_id: currentMandateId }),
  });
  const order = await res.json();
  currentOrderId = order.order_id;
  document.getElementById("orderBox").innerHTML = `Order ${order.order_id} — status: <b>${order.status}</b><br>Pay link: ${order.payment_link_url}`;
}

async function triggerWebhook(outcome) {
  if (!currentOrderId) return;
  const res = await fetch(`${API}/demo/trigger-webhook?order_id=${currentOrderId}&outcome=${outcome}`, { method: "POST" });
  const order = await res.json();
  document.getElementById("orderBox").innerHTML = `Order ${order.order_id} — status: <b>${order.status}</b>` +
    (order.failure_reason ? `<br>Reason: ${order.failure_reason}` : "");
}

async function refreshAudit() {
  const res = await fetch(`${API}/audit`);
  const data = await res.json();
  document.getElementById("chainStatus").innerHTML =
    `Hash chain: <span class="badge ${data.chain.valid ? 'pass' : 'fail'}">${data.chain.valid ? 'VALID (' + data.chain.entries + ' entries)' : 'TAMPERED'}</span>`;
  document.getElementById("auditLog").textContent = JSON.stringify(data.events.slice(0, 20), null, 2);
  const analytics = await (await fetch(`${API}/analytics`)).json();
  document.getElementById("analyticsBox").textContent = JSON.stringify(analytics, null, 2);
}

async function runJourneyB() {
  const log = document.getElementById("journeyBLog");
  const mcpSession = "mcp_" + Date.now();
  const append = (line) => { log.textContent += line + "\\n"; };
  log.textContent = "";

  append("[agent] list_products()");
  const products = (await (await fetch(`${API}/mcp/list_products`)).json()).products;
  append(`  -> ${products.length} products found`);

  append("[agent] budget ₹1200 -> choosing kr-filter-500 (₹450) + kr-steel-filter (₹650) = ₹1100");
  await fetch(`${API}/mcp/add_to_cart`, { method: "POST", headers: {"Content-Type":"application/json"},
    body: JSON.stringify({ session_id: mcpSession, sku: "kr-filter-500" }) });
  await fetch(`${API}/mcp/add_to_cart`, { method: "POST", headers: {"Content-Type":"application/json"},
    body: JSON.stringify({ session_id: mcpSession, sku: "kr-steel-filter" }) });
  append("[agent] add_to_cart(kr-filter-500), add_to_cart(kr-steel-filter)");

  const mandate = await (await fetch(`${API}/mcp/create_checkout_mandate`, { method: "POST", headers: {"Content-Type":"application/json"},
    body: JSON.stringify({ session_id: mcpSession, buyer_ref: "external_ai_agent" }) })).json();
  append(`[agent] create_checkout_mandate() -> ${mandate.mandate_id} (${mandate.status}, ₹${(mandate.total_paise/100).toFixed(0)})`);

  const result = await (await fetch(`${API}/mcp/confirm_and_pay`, { method: "POST", headers: {"Content-Type":"application/json"},
    body: JSON.stringify({ mandate_id: mandate.mandate_id }) })).json();
  append(`[agent] confirm_and_pay(${mandate.mandate_id}) -> order ${result.order.order_id} (${result.order.status})`);
  append("No human typed a single character in this flow.");
}
</script>
</body>
</html>
'''

with open("kaapi_copilot/frontend/index.html", "w") as f:
    f.write(frontend_html)

print("Frontend written to kaapi_copilot/frontend/index.html")
print("Open it in a browser while the FastAPI backend runs on http://localhost:8000")


Frontend written to kaapi_copilot/frontend/index.html

Open it in a browser while the FastAPI backend runs on http://localhost:8000


## 19. Tests

`pytest` suite covering the hard constraints called out in the brief: catalog price re-verification (mismatch blocks, never silently corrects), spend-cap breach produces a hard `blocked` mandate, only `confirmed` mandates can reach checkout, the audit hash-chain detects tampering, and the UPI decline path never marks an order `paid`.

In [ ]:
test_py = '''"""
pytest suite for the Kaapi Copilot guardrail contract. Run from kaapi_copilot/backend:
    pytest ../tests/test_guardrails.py
"""
import os
import sys
import pytest

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "backend"))

OVER_CAP_SKUS = ["kr-subscription", "kr-dripper", "kr-frother", "kr-steel-filter", "kr-arabica-250"]  # 318000 paise > 300000 cap


@pytest.fixture(autouse=True)
def fresh_services(tmp_path, monkeypatch):
    """Reload all app modules against a throwaway sqlite db per test."""
    for mod in list(sys.modules):
        if mod.startswith("app."):
            del sys.modules[mod]
    monkeypatch.setenv("KAAPI_DB_PATH", str(tmp_path / "test.db"))
    yield


def test_price_mismatch_blocks_mandate():
    from app.services.mandate_engine import mandate_engine
    mandate = mandate_engine.build_mandate(
        "sess1", "buyer1", ["kr-filter-500"], "test",
        agent_stated_prices={"kr-filter-500": 1},  # deliberately wrong
    )
    assert mandate.status == "blocked"
    assert "catalog says" in mandate.block_reason


def test_transaction_cap_breach_is_hard_blocked():
    from app.services.mandate_engine import mandate_engine
    mandate = mandate_engine.build_mandate("sess2", "buyer2", OVER_CAP_SKUS, "test")
    assert mandate.status == "blocked"
    checks = {c.rule: c.status for c in mandate.policy_checks}
    assert checks["transaction_spend_cap"] == "fail"


def test_only_confirmed_mandate_can_checkout():
    from app.services.mandate_engine import mandate_engine, GuardrailError
    from app.services.order_service import order_service
    mandate = mandate_engine.build_mandate("sess3", "buyer3", ["kr-filter-500"], "test")
    order_service.register_mandate(mandate)
    with pytest.raises(GuardrailError):
        order_service.checkout(mandate)  # never confirmed


def test_blocked_mandate_cannot_be_confirmed():
    from app.services.mandate_engine import mandate_engine, GuardrailError
    mandate = mandate_engine.build_mandate("sess4", "buyer4", OVER_CAP_SKUS, "test")
    with pytest.raises(GuardrailError):
        mandate_engine.confirm_mandate(mandate, method="buyer_tap")


def test_upi_decline_never_marks_order_paid():
    from app.services.mandate_engine import mandate_engine
    from app.services.order_service import order_service
    from app.providers.payment.mock import mock_payment_provider

    mandate = mandate_engine.build_mandate("sess5", "buyer5", ["kr-frother"], "test")
    order_service.register_mandate(mandate)
    mandate_engine.confirm_mandate(mandate, method="buyer_tap")
    order = order_service.checkout(mandate)

    if order.payment_link_id is None:
        raise ValueError("Order has no payment link")
    wh = mock_payment_provider.simulate_webhook(order.payment_link_id, "failure")
    updated = order_service.handle_webhook_event(wh["event"], wh["payload"])

    assert updated.status == "payment_failed"
    assert updated.status != "paid"
    assert order_service.get_held_cart("sess5") is not None


def test_audit_hash_chain_detects_tampering():
    from app.services.audit_trail import audit_trail
    audit_trail.log("event_a", "sessX", {"k": 1})
    audit_trail.log("event_b", "sessX", {"k": 2})
    assert audit_trail.verify_chain()["valid"] is True

    import sqlite3
    conn = sqlite3.connect(audit_trail.db_path)
    conn.execute("UPDATE audit_log SET payload = ? WHERE event_type = ?", ('{"k": 999}', "event_a"))
    conn.commit()
    conn.close()

    result = audit_trail.verify_chain()
    assert result["valid"] is False
'''

os.makedirs("kaapi_copilot/tests", exist_ok=True)
with open("kaapi_copilot/tests/test_guardrails.py", "w") as f:
    f.write(test_py)

import subprocess, sys, os
result = subprocess.run(
    [sys.executable, "-m", "pytest", "kaapi_copilot/tests/test_guardrails.py", "-v"],
    cwd=os.getcwd(), capture_output=True, text=True,
)
print(result.stdout[-3000:])
print(result.stderr[-1500:])


============================= test session starts ==============================

platform linux -- Python 3.12.12, pytest-9.1.1, pluggy-1.6.0 -- /opt/conda/bin/python

cachedir: .pytest_cache

rootdir: /home/jovyan/projects/Kaapi_Copilot_AI_Commerce_Platform-2026_08_26-03_33_20

plugins: anyio-4.14.0, time-machine-2.19.0

collecting ... collected 6 items



kaapi_copilot/tests/test_guardrails.py::test_price_mismatch_blocks_mandate PASSED [ 16%]

kaapi_copilot/tests/test_guardrails.py::test_transaction_cap_breach_is_hard_blocked PASSED [ 33%]

kaapi_copilot/tests/test_guardrails.py::test_only_confirmed_mandate_can_checkout PASSED [ 50%]

kaapi_copilot/tests/test_guardrails.py::test_blocked_mandate_cannot_be_confirmed PASSED [ 66%]

kaapi_copilot/tests/test_guardrails.py::test_upi_decline_never_marks_order_paid PASSED [ 83%]

kaapi_copilot/tests/test_guardrails.py::test_audit_hash_chain_detects_tampering PASSED [100%]



============================== 6 passed in 0.22s =================

## 20. README

Full project README: pitch, architecture diagram, setup (mock mode by default; env vars to flip to live Anthropic/Razorpay test mode), guardrail explanation, demo flow, API endpoint list, and known limitations.

In [5]:
readme = '''# Kaapi Copilot — Agentic Storefront for Kaapi Roasters

Kaapi Copilot turns a small D2C filter-coffee brand's storefront into an AI-agent-operated,
transactable business: a Groq-powered shopping agent runs discovery → recommendation →
one catalog-defined upsell → a guardrail-gated **Purchase Mandate** → Razorpay checkout for
human buyers (Journey A), and the identical catalog/cart/checkout is exposed as MCP-style
tools so an **external AI agent can shop and pay with no human in the loop** (Journey B).
In a smoke-test run, the agent-assisted average order value was **₹1,100 vs. a ₹450 baseline
single-item purchase — a 144% AOV lift** driven entirely by the one-upsell-per-turn rule.

Runs fully in **MOCK MODE** out of the box — no API keys required — and switches to live
Groq + Razorpay test-mode calls the moment real credentials are present.

## Architecture

```mermaid
flowchart LR
    subgraph Buyers
        H["Human buyer (chat UI)"]
        X["External AI agent (Journey B)"]
    end

    subgraph Storefront["Agentic Storefront"]
        AGENT["Shopping & Upsell Agent (Groq, tool calling)"]
        CAT["Catalog Service (REST + MCP resource)"]
        GUARD["Guardrail / Mandate Engine"]
        AUDIT[("Audit Trail Store")]
        MCPS["MCP Server (list_products, add_to_cart, confirm_and_pay)"]
    end

    subgraph RZP["Razorpay (Test Mode)"]
        API["Orders / Payment Links (via Razorpay MCP)"]
        WEBHOOK["Webhook: payment.captured / payment.failed"]
    end

    H --> AGENT
    X --> MCPS --> AGENT
    AGENT --> CAT
    AGENT --> GUARD
    GUARD -- "confirmed mandate only" --> API
    GUARD -- "every check + call" --> AUDIT
    API --> WEBHOOK --> GUARD
    GUARD --> AGENT
```

## Provider abstraction (mock vs. real, swappable without touching business logic)

| Layer | Mock (default) | Real |
|---|---|---|
| Payments | `MockPaymentProvider` — in-memory orders/links/webhooks, recognizes `success@razorpay` / `failure@razorpay` | `RazorpayPaymentProvider` — official `razorpay` SDK, test-mode keys |
| Shopping agent | `MockShoppingAgent` — deterministic keyword matching against the catalog | `GroqShoppingAgent` — Groq Chat Completions API, tool calling, bounded loop |

Selected via env vars, with automatic fallback to mock if secrets are missing:

```
PAYMENT_MODE=mock            # or "razorpay"
AGENT_MODE=mock               # or "groq"
GROQ_API_KEY=...
RAZORPAY_KEY_ID=...
RAZORPAY_KEY_SECRET=...
RAZORPAY_WEBHOOK_SECRET=...
```

## Setup & run locally

```bash
cd kaapi_copilot/backend
pip install -r requirements.txt
uvicorn app.api.main:app --reload --port 8000
```

Open `kaapi_copilot/frontend/index.html` directly in a browser (it calls
`http://localhost:8000/api/*`). No build step, no separate catalog seed script —
`SEED_PRODUCTS` in `app/data/catalog_data.py` loads deterministically on import.

Run tests:
```bash
pytest kaapi_copilot/tests/test_guardrails.py -v
```

## How the guardrails work

`MandateEngine.build_mandate()` is plain code, not a prompt instruction:

1. **Every line price is re-read from the catalog**, never taken from agent-stated text.
   A mismatch is a hard block (`price_matches_catalog: fail`) — never silently corrected.
2. **Session + per-transaction spend caps** are checked before any Razorpay call. A breach
   is a hard stop producing a `blocked` mandate — never a soft warning.
3. **Every mandate is written to the hash-chained audit trail before any Razorpay call.**
   `AuditTrail.verify_chain()` recomputes every SHA-256 link; tampering is detectable.
4. **Only a `confirmed` mandate may reach `OrderService.checkout()`** — the sole code path
   allowed to call `create_order` / `create_payment_link`. Journey A confirms via an explicit
   buyer tap; Journey B confirms via the explicit `confirm_and_pay` MCP tool call. There is no
   route from raw agent/LLM output straight to a payment call.
5. **Webhook-driven state only.** Orders flip to `paid` only from `payment.captured`; a
   `payment.failed` webhook holds the cart for 10 minutes and never marks the order paid.

## Demo flow

1. Chat: *"I want good filter coffee, nothing fancy"* → agent recommends Filter Coffee
   Powder (₹450) and proposes the catalog-defined upsell, the Steel Filter Set (₹650).
2. *"sure, add it"* → cart now ₹1,100.
3. Build Mandate → policy checks all `pass` → Proceed to Pay → checkout → simulate
   `success@razorpay` → order `paid`.
4. Failure demo: same flow, simulate `failure@razorpay` → order `payment_failed`, cart held,
   never `paid`.
5. Guardrail demo: push a cart over ₹3,000 → mandate `blocked` before any Razorpay call.
6. Journey B: click "Run Journey B demo" — an external agent calls `list_products`,
   `add_to_cart`, `create_checkout_mandate`, `confirm_and_pay` with zero human typing.
7. Ops panel: live audit trail + hash-chain validity + revenue analytics.

## API endpoints

- `GET /api/health` — mode summary
- `POST /api/chat` — Journey A conversational turn
- `POST /api/mandates/build`, `POST /api/mandates/confirm` — guardrail-gated mandate lifecycle
- `POST /api/checkout` — confirmed-mandate-only order + payment link creation
- `POST /api/webhooks/razorpay` — signature-verified webhook receiver
- `POST /api/demo/trigger-webhook` — demo-only mock webhook trigger
- `GET /api/audit`, `GET /api/analytics` — Ops panel data
- `GET /api/mcp/list_products`, `GET /api/mcp/get_product`, `POST /api/mcp/add_to_cart`,
  `GET /api/mcp/get_cart`, `POST /api/mcp/create_checkout_mandate`,
  `POST /api/mcp/confirm_and_pay` — Journey B MCP-style tool surface

## Switching to live Groq + Razorpay test mode

1. Set `GROQ_API_KEY` → `AGENT_MODE=groq` activates `GroqShoppingAgent`.
2. Generate Razorpay **test-mode** keys (Dashboard → Test Mode → Settings → API Keys), set
   `RAZORPAY_KEY_ID` / `RAZORPAY_KEY_SECRET` / `RAZORPAY_WEBHOOK_SECRET`, then
   `PAYMENT_MODE=razorpay`. Point Razorpay's webhook settings at
   `<public_url>/api/webhooks/razorpay`.
3. Use UPI test handles `success@razorpay` / `failure@razorpay` on the real payment link to
   exercise the happy path and the graceful-failure path against Razorpay's real test-mode
   infrastructure.

## Known limitations

- All state (sessions, orders, mandates, MCP carts) is in-memory except the audit log
  (SQLite); restarting the backend clears carts/sessions.
- `GroqShoppingAgent` and `RazorpayPaymentProvider` are implemented but not exercised in
  this environment (no live keys); mock-mode is the verified default path.
- Recurring/subscription billing (`RecurringMandateService`) simulates cycles on demand
  rather than running on a real scheduler.
- No authentication/authorization on the API — this is a demo prototype, not production.
- Campaign orchestrator (abandoned-cart nudges) from the stretch goals is not implemented.
'''

with open("kaapi_copilot/README.md", "w") as f:
    f.write(readme)

print("README written to kaapi_copilot/README.md")
print(f"Length: {len(readme)} characters")


README written to kaapi_copilot/README.md
Length: 6809 characters
